<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_5/All_Examples_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Тема 4. Предобработка данных для RAG (Ingestion)

In [ ]:
# Установка только необходимых библиотек
!pip install pypdf pdfplumber

import os
import json
import logging
from datetime import datetime
from typing import Dict, List
from pathlib import Path

import pdfplumber
from pypdf import PdfReader

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def extract_pdf_text(file_path: str) -> str:
    """Извлекает текст из PDF с fallback."""
    text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        if text.strip():
            return text
    except Exception as e:
        logger.warning(f"pdfplumber не сработал для {file_path}: {e}")

    try:
        with open(file_path, "rb") as f:
            reader = PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        logger.error(f"Не удалось извлечь текст из {file_path}: {e}")
        return ""

    return text.strip()


def clean_text(text: str) -> str:
    """Очищает текст от шума."""
    import re
    if not text:
        return ""
    # Удаляем управляющие символы
    text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', '', text)
    # Нормализуем пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def get_pdf_metadata(file_path: str) -> Dict:
    """Извлекает метаданные из PDF и файловой системы."""
    stat = os.stat(file_path)
    metadata = {
        "source": os.path.basename(file_path),
        "file_path": str(file_path),
        "doc_id": Path(file_path).stem,
        "created_date": datetime.fromtimestamp(stat.st_ctime).isoformat(),
        "modified_date": datetime.fromtimestamp(stat.st_mtime).isoformat(),
        "page_number": 0,
        "section": "",
        "tags": [],
        "department": "",
        "is_active": True
    }
    try:
        with open(file_path, "rb") as f:
            reader = PdfReader(f)
            info = reader.metadata
            if info:
                metadata["author"] = str(info.get('/Author', ''))
                metadata["title"] = str(info.get('/Title', ''))
    except Exception as e:
        logger.warning(f"Не удалось извлечь метаданные PDF для {file_path}: {e}")
    return metadata


def process_pdf_directory(input_dir: str, output_json: str) -> List[Dict]:
    """Обрабатывает все PDF в директории и сохраняет результат в JSON."""
    results = []
    pdf_files = list(Path(input_dir).glob("**/*.pdf"))

    if not pdf_files:
        logger.warning(f"PDF файлы не найдены в {input_dir}")
        return results

    logger.info(f"Найдено {len(pdf_files)} PDF файлов")

    for pdf_path in pdf_files:
        logger.info(f"Обработка: {pdf_path}")
        text = extract_pdf_text(str(pdf_path))
        if not text:
            logger.warning(f"Пропущен (пустой текст): {pdf_path}")
            continue

        cleaned_text = clean_text(text)
        metadata = get_pdf_metadata(str(pdf_path))

        results.append({
            "text": cleaned_text,
            "metadata": metadata
        })
        logger.info(f"Добавлен: {pdf_path} (длина текста: {len(cleaned_text)} символов)")

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    logger.info(f"Сохранено {len(results)} документов в {output_json}")
    return results


if __name__ == "__main__":
    process_pdf_directory(
        input_dir="./documents",
        output_json="./extracted_data.json"
    )

#Тема 5. Чанкинг (разбиение текста)

In [ ]:
# ================================================================
# Тема 5. Чанкинг (разбиение текста)
# Использует данные из extracted_data.json (результат раздела 4)
# ================================================================

import json
import re
from typing import List, Dict
from pathlib import Path

class RecursiveTextSplitter:
    """
    Рекурсивный сплиттер с иерархией разделителей.
    Разбивает текст на чанки, сохраняя структуру документа.
    """
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        # Иерархия разделителей: от крупных к мелким
        self.separators = ["\n\n", "\n", ". ", "! ", "? ", ", ", " "]

    def split_document(self, text: str, metadata: Dict) -> List[Dict]:
        """
        Разбивает текст на чанки и добавляет метаданные к каждому чанку.
        Возвращает список словарей: {"text": str, "metadata": dict}
        """
        chunks_text = self._split_text(text)
        chunks_with_meta = []
        for i, chunk_text in enumerate(chunks_text):
            chunk_meta = metadata.copy()
            chunk_meta["chunk_index"] = i
            chunk_meta["chunk_length"] = len(chunk_text)
            chunks_with_meta.append({
                "text": chunk_text,
                "metadata": chunk_meta
            })
        return chunks_with_meta

    def _split_text(self, text: str) -> List[str]:
        """Разбивает текст на чанки (внутренний метод)."""
        if not text:
            return []
        chunks = []
        current_chunk = []
        current_len = 0

        # Разбиваем текст рекурсивно по разделителям
        segments = self._split_by_separators(text, self.separators)

        for segment in segments:
            seg_len = len(segment)
            # Если текущий чанк + новый сегмент превышает max_size и чанк не пуст
            if current_len + seg_len > self.chunk_size and current_chunk:
                chunks.append("".join(current_chunk).strip())
                # Извлекаем overlap из предыдущего чанка
                overlap_text = self._get_overlap("".join(current_chunk), self.chunk_overlap)
                current_chunk = [overlap_text]
                current_len = len(overlap_text)

            current_chunk.append(segment)
            current_len += seg_len

        if current_chunk:
            chunks.append("".join(current_chunk).strip())

        return chunks

    def _split_by_separators(self, text: str, separators: List[str]) -> List[str]:
        """Рекурсивно разбивает текст по разделителям."""
        if not text:
            return []

        separator = separators[0]
        remaining_seps = separators[1:]

        if not remaining_seps:
            # Последний разделитель — пробел
            return text.split(separator)

        parts = text.split(separator)
        result = []
        for i, part in enumerate(parts):
            if len(part) <= self.chunk_size:
                result.append(part)
            else:
                # Рекурсивно разбиваем более мелким разделителем
                sub_parts = self._split_by_separators(part, remaining_seps)
                result.extend(sub_parts)

            if i < len(parts) - 1:
                result.append(separator)  # возвращаем разделитель обратно

        return result

    def _get_overlap(self, text: str, overlap_len: int) -> str:
        """Возвращает последние `overlap_len` символов текста."""
        return text[-overlap_len:] if len(text) > overlap_len else text


def load_extracted_data(json_path: str = "./extracted_data.json") -> List[Dict]:
    """Загружает данные, полученные в разделе 4."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data


def process_chunking(input_json: str = "./extracted_data.json",
                     output_json: str = "./chunks_data.json",
                     chunk_size: int = 500,
                     chunk_overlap: int = 50) -> List[Dict]:
    """
    Загружает данные из input_json, применяет чанкинг и сохраняет в output_json.
    Возвращает список всех чанков с метаданными.
    """
    # Загрузка данных из раздела 4
    documents = load_extracted_data(input_json)
    print(f"Загружено {len(documents)} документов.")

    splitter = RecursiveTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    all_chunks = []

    for doc in documents:
        text = doc["text"]
        metadata = doc["metadata"]
        if not text:
            continue
        chunks = splitter.split_document(text, metadata)
        all_chunks.extend(chunks)

    # Сохранение результатов
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)

    print(f"Создано {len(all_chunks)} чанков из {len(documents)} документов.")
    print(f"Результат сохранён в {output_json}")
    return all_chunks


def compare_chunk_sizes(input_json: str = "./extracted_data.json"):
    """
    Сравнивает три размера чанков на первом документе из загруженных данных.
    Выводит статистику для каждого размера.
    """
    documents = load_extracted_data(input_json)
    if not documents:
        print("Нет данных для эксперимента.")
        return

    sample_doc = documents[0]
    text = sample_doc["text"]
    metadata = sample_doc["metadata"]

    sizes = [200, 500, 1000]
    overlaps = [20, 50, 100]  # 10% от размера

    print("=" * 60)
    print(f"Эксперимент на документе: {metadata.get('source', 'unknown')}")
    print(f"Длина текста: {len(text)} символов")
    print("=" * 60)

    for size, overlap in zip(sizes, overlaps):
        splitter = RecursiveTextSplitter(chunk_size=size, chunk_overlap=overlap)
        chunks = splitter.split_document(text, metadata)
        print(f"\nРазмер чанка: {size}, overlap: {overlap}")
        print(f"  Количество чанков: {len(chunks)}")
        if chunks:
            print(f"  Пример первого чанка (первые 100 символов):")
            print(f"    {chunks[0]['text'][:100]}...")
        lengths = [len(c['text']) for c in chunks]
        print(f"  Средняя длина: {sum(lengths)/len(lengths):.0f} символов")
        print(f"  Минимальная: {min(lengths)}, максимальная: {max(lengths)}")


if __name__ == "__main__":
    # Шаг 1: применить чанкинг с параметрами по умолчанию
    chunks = process_chunking(
        input_json="./extracted_data.json",
        output_json="./chunks_data.json",
        chunk_size=500,
        chunk_overlap=50
    )

    # Шаг 2: провести сравнение размеров
    compare_chunk_sizes()

5.4.2. Использование LangChain (альтернативный вариант)

In [ ]:
!pip install langchain-text-splitters

In [ ]:
# ================================================================
# Тема 5. Чанкинг (разбиение текста) с использованием LangChain
# Альтернативный вариант к разделу 5.4.1
# Использует данные из extracted_data.json (результат раздела 4)
# ================================================================

import json
from typing import List, Dict

from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_extracted_data(json_path: str = "./extracted_data.json") -> List[Dict]:
    """Загружает данные, полученные в разделе 4."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data


def split_with_langchain(input_json: str = "./extracted_data.json",
                         output_json: str = "./chunks_data_langchain.json",
                         chunk_size: int = 500,
                         chunk_overlap: int = 50) -> List[Dict]:
    """
    Загружает данные из input_json, применяет чанкинг с помощью LangChain
    и сохраняет результат в output_json.
    Возвращает список всех чанков с метаданными.
    """
    # 1. Загрузка данных из раздела 4
    documents = load_extracted_data(input_json)
    print(f"Загружено {len(documents)} документов.")

    # 2. Настройка сплиттера LangChain
    # Используем RecursiveCharacterTextSplitter с теми же параметрами,
    # что и в нашей ручной реализации
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        # Иерархия разделителей: от крупных к мелким
        separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " "],
        length_function=len,  # считаем длину в символах
    )

    all_chunks = []

    # 3. Обработка каждого документа
    for doc in documents:
        text = doc["text"]
        metadata = doc["metadata"]

        if not text:
            continue

        # Разбиваем текст на чанки с помощью LangChain
        chunks_text = splitter.split_text(text)

        # Добавляем метаданные к каждому чанку
        for i, chunk_text in enumerate(chunks_text):
            chunk_meta = metadata.copy()
            chunk_meta["chunk_index"] = i
            chunk_meta["chunk_length"] = len(chunk_text)
            all_chunks.append({
                "text": chunk_text,
                "metadata": chunk_meta
            })

    # 4. Сохранение результатов
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)

    print(f"Создано {len(all_chunks)} чанков из {len(documents)} документов.")
    print(f"Результат сохранён в {output_json}")
    return all_chunks


def compare_chunk_sizes_langchain(input_json: str = "./extracted_data.json"):
    """
    Сравнивает три размера чанков на первом документе из загруженных данных.
    Использует LangChain для сплиттинга.
    """
    documents = load_extracted_data(input_json)
    if not documents:
        print("Нет данных для эксперимента.")
        return

    sample_doc = documents[0]
    text = sample_doc["text"]
    metadata = sample_doc["metadata"]

    sizes = [200, 500, 1000]
    overlaps = [20, 50, 100]  # 10% от размера

    print("=" * 60)
    print(f"Эксперимент (LangChain) на документе: {metadata.get('source', 'unknown')}")
    print(f"Длина текста: {len(text)} символов")
    print("=" * 60)

    for size, overlap in zip(sizes, overlaps):
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=size,
            chunk_overlap=overlap,
            separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " "],
            length_function=len,
        )
        chunks_text = splitter.split_text(text)
        print(f"\nРазмер чанка: {size}, overlap: {overlap}")
        print(f"  Количество чанков: {len(chunks_text)}")
        if chunks_text:
            print(f"  Пример первого чанка (первые 100 символов):")
            print(f"    {chunks_text[0][:100]}...")
        lengths = [len(c) for c in chunks_text]
        if lengths:
            print(f"  Средняя длина: {sum(lengths)/len(lengths):.0f} символов")
            print(f"  Минимальная: {min(lengths)}, максимальная: {max(lengths)}")


if __name__ == "__main__":
    # Шаг 1: применить чанкинг с параметрами по умолчанию
    chunks = split_with_langchain(
        input_json="./extracted_data.json",
        output_json="./chunks_data_langchain.json",
        chunk_size=500,
        chunk_overlap=50
    )

    # Шаг 2: провести сравнение размеров
    compare_chunk_sizes_langchain()

# Тема 6. Векторизация и эмбеддинги

6.2.2. BGE (BAAI/bge)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Загрузка модели
model = SentenceTransformer('BAAI/bge-base-en-v1.5')

# Важно: для BGE нужно использовать префиксы!
# Для запросов (queries) — "query: "
# Для документов (passages) — "passage: "

queries = [
    "query: Какие налоги платят самозанятые?",
    "query: Ставка налога на прибыль"
]

documents = [
    "passage: Налог на прибыль организаций регулируется главой 25 НК РФ.",
    "passage: Ставка налога на прибыль составляет 20%.",
    "passage: Самозанятые платят налог на профессиональный доход."
]

# Генерация эмбеддингов
query_embeddings = model.encode(queries, normalize_embeddings=True)
doc_embeddings = model.encode(documents, normalize_embeddings=True)

# Вычисление косинусного сходства
from sklearn.metrics.pairwise import cosine_similarity
scores = cosine_similarity(query_embeddings, doc_embeddings)

print("Матрица сходства (запросы × документы):")
print(scores)

6.2.3. OpenAI Embeddings

In [ ]:
!pip install openai
import openai
import numpy as np
from typing import List

# Установите ваш API-ключ
openai.api_key = "your-api-key-here"

def get_openai_embeddings(texts: List[str], model: str = "text-embedding-3-small") -> np.ndarray:
    """
    Генерирует эмбеддинги через OpenAI API.
    """
    # Для моделей text-embedding-3 нужно уменьшить размерность (опционально)
    # dimensions=1024  # можно уменьшить до 1024 для экономии
    response = openai.embeddings.create(
        model=model,
        input=texts,
        # dimensions=1024  # раскомментируйте, если хотите уменьшить размерность
    )
    embeddings = np.array([item.embedding for item in response.data])
    return embeddings

# Пример использования
texts = [
    "Налог на прибыль организаций регулируется главой 25 НК РФ.",
    "Ставка налога на прибыль составляет 20%."
]

embeddings = get_openai_embeddings(texts, model="text-embedding-3-small")
print(f"Размерность: {embeddings.shape[1]}")
print(f"Форма: {embeddings.shape}")

6.2.4. Русскоязычные модели

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Работающая русская модель
model = SentenceTransformer('cointegrated/rubert-tiny2')

texts = [
    "Налог на прибыль организаций регулируется главой 25 НК РФ.",
    "Ставка налога на прибыль составляет 20%.",
    "Самозанятые платят налог на профессиональный доход."
]

embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print(f"Размерность: {embeddings.shape[1]}")
print(f"Вектор для первого текста (первые 5 значений): {embeddings[0][:5]}")

6.3.2. Генерация эмбеддингов с помощью sentence‑transformers

In [ ]:
!pip install --upgrade sentence-transformers pillow transformers
# Установка только нужных библиотек для извлечения текста
!pip install pypdf pdfplumber

# Установка sentence-transformers и фикс Pillow
!pip install sentence-transformers
!pip uninstall pillow -y
!pip install pillow==10.4.0

In [ ]:
# Теперь импорты должны работать
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List

class EmbeddingGenerator:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", device: str = "cpu"):
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name
        self.dimension = self.model.get_sentence_embedding_dimension()
        print(f"Модель: {model_name}, размерность: {self.dimension}, устройство: {device}")

    def encode_batch(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        return self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

# Проверка
generator = EmbeddingGenerator()
texts = ["Пример текста", "Ещё один документ"]
embeddings = generator.encode_batch(texts)
print(embeddings.shape)

6.3.3. Кэширование эмбеддингов на диск

In [ ]:
import pickle
import os
import numpy as np
from typing import Dict, List, Any
from sentence_transformers import SentenceTransformer

class EmbeddingGenerator:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", device: str = None):
        # Если device не указан, определяем автоматически
        if device is None:
            import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name
        self.dimension = self.model.get_sentence_embedding_dimension()
        print(f"Модель: {model_name}, размерность: {self.dimension}, устройство: {device}")

    def encode_batch(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        return self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

class EmbeddingCache:
    def __init__(self, cache_dir: str = "./embedding_cache"):
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)

    def get_cache_path(self, model_name: str, chunk_id: int) -> str:
        return os.path.join(self.cache_dir, f"{model_name}_{chunk_id}.pkl")

    def save_embeddings(self, model_name: str, chunk_id: int, embeddings: np.ndarray, metadata: Dict = None):
        cache_path = self.get_cache_path(model_name, chunk_id)
        data = {"embeddings": embeddings, "metadata": metadata}
        with open(cache_path, "wb") as f:
            pickle.dump(data, f)

    def load_embeddings(self, model_name: str, chunk_id: int) -> Dict:
        cache_path = self.get_cache_path(model_name, chunk_id)
        if os.path.exists(cache_path):
            with open(cache_path, "rb") as f:
                return pickle.load(f)
        return None

    def exists(self, model_name: str, chunk_id: int) -> bool:
        return os.path.exists(self.get_cache_path(model_name, chunk_id))

# Пример использования
generator = EmbeddingGenerator("all-MiniLM-L6-v2")  # device определится автоматически
cache = EmbeddingCache("./embedding_cache")

texts = ["Это текст 1", "Это текст 2"]
chunk_id = 1

if cache.exists(generator.model_name, chunk_id):
    data = cache.load_embeddings(generator.model_name, chunk_id)
    embeddings = data["embeddings"]
    print("Эмбеддинги загружены из кэша")
else:
    embeddings = generator.encode_batch(texts)
    cache.save_embeddings(generator.model_name, chunk_id, embeddings, {"texts": texts})
    print("Эмбеддинги сгенерированы и сохранены в кэш")

print(f"Форма эмбеддингов: {embeddings.shape}")

Тема 7. Векторные базы данных

In [ ]:
!pip install chromadb

In [ ]:
import chromadb
from chromadb.config import Settings
import json
from sentence_transformers import SentenceTransformer

def load_documents(json_path: str = "./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def clean_metadata(metadata: dict) -> dict:
    cleaned = {}
    for key, value in metadata.items():
        if isinstance(value, list):
            if len(value) == 0:
                continue
            value = [v for v in value if v is not None]
            if len(value) == 0:
                continue
        elif value is None or value == "":
            continue
        cleaned[key] = value
    return cleaned

# 1. Инициализация
client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=Settings(anonymized_telemetry=False)
)

collection = client.get_or_create_collection(
    name="documents",
    metadata={"hnsw:space": "cosine"}
)

# 2. Загрузка данных
documents = load_documents("./extracted_data.json")

texts = [doc["text"] for doc in documents]
metadatas = [clean_metadata(doc["metadata"]) for doc in documents]
ids = [f"doc_{i:04d}" for i in range(len(documents))]

# 3. Генерация эмбеддингов (если их нет)
model = SentenceTransformer("cointegrated/rubert-tiny2")
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True).tolist()

# 4. Добавление
collection.add(documents=texts, embeddings=embeddings, metadatas=metadatas, ids=ids)
print(f"✅ Добавлено {len(documents)} документов.")

# 5. Поиск БЕЗ ФИЛЬТРА
query = "налог на прибыль"
query_emb = model.encode([query], normalize_embeddings=True).tolist()

results = collection.query(
    query_embeddings=query_emb,
    n_results=3
)

print(f"\n🔍 Найдено: {len(results['documents'][0])} документов")
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(f"  - {doc[:150]}... (источник: {meta.get('source', 'unknown')})")


### 7.6.2. FAISS – максимальная скорость, метаданные отдельно




In [ ]:
# ================================================================
# FAISS: исправленная версия (адаптивная)
# ================================================================

import faiss
import numpy as np
import json
from sentence_transformers import SentenceTransformer

def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# Загрузка данных
docs = load_documents()
texts = [d["text"] for d in docs]
metadatas = [d["metadata"] for d in docs]

if not texts:
    print("Нет документов для индексации")
    exit()

model = SentenceTransformer("cointegrated/rubert-tiny2")
embeddings = model.encode(texts, normalize_embeddings=True).astype('float32')
dim = embeddings.shape[1]
n_vectors = len(embeddings)

# Выбор типа индекса в зависимости от количества векторов
if n_vectors < 100:
    # Для малого числа документов используем точный поиск (Flat)
    index = faiss.IndexFlatIP(dim)  # IP = Inner Product (для нормализованных векторов даёт косинусное сходство)
    index.add(embeddings)
    print(f"Используется IndexFlatIP (точный поиск) для {n_vectors} векторов")
else:
    # Для больших данных – IVF
    nlist = min(100, max(1, n_vectors // 10))
    quantizer = faiss.IndexFlatIP(dim)
    index = faiss.IndexIVFFlat(quantizer, dim, nlist)
    index.train(embeddings)
    index.add(embeddings)
    index.nprobe = min(5, nlist)
    print(f"Используется IndexIVFFlat с nlist={nlist}, nprobe={index.nprobe}")

# Поиск
query = "налог на прибыль"
query_emb = model.encode([query], normalize_embeddings=True).astype('float32')

distances, indices = index.search(query_emb, min(3, n_vectors))

print("Результаты поиска FAISS:")
for idx, dist in zip(indices[0], distances[0]):
    if idx >= 0 and idx < len(texts):
        print(f"  - {metadatas[idx].get('source', 'unknown')}: {texts[idx][:150]}... (сходство: {dist:.4f})")
    else:
        print(f"  - индекс {idx} вне диапазона")



### 7.6.3. Pinecone – облачный сервис (требуется API-ключ)


In [ ]:
!pip install --upgrade pinecone

In [ ]:
# ================================================================
# PINECONE – ФИНАЛЬНЫЙ РАБОЧИЙ КОД (регион us-east-1)
# ================================================================

import json
import time
from sentence_transformers import SentenceTransformer
import pinecone
from google.colab import userdata

# 1. Загрузка документов
def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# 2. Инициализация Pinecone
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
pc = pinecone.Pinecone(api_key=PINECONE_API_KEY)

# 3. Параметры индекса (исправленный регион)
index_name = "rag-docs"
dimension = 312
metric = "cosine"
cloud = "aws"          # Используем AWS
region = "us-east-1"   # Стандартный регион для новых аккаунтов

# 4. Создание индекса (если не существует)
existing_indexes = [idx.name for idx in pc.list_indexes()]
if index_name not in existing_indexes:
    print(f"Создание индекса {index_name} в регионе {region}...")
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric=metric,
        spec=pinecone.ServerlessSpec(cloud=cloud, region=region)
    )
    # Ожидание готовности (до 2 минут)
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(10)
        print("Ожидание готовности индекса...")
    print("✅ Индекс готов.")
else:
    print(f"ℹ️ Индекс {index_name} уже существует.")

index = pc.Index(index_name)

# 5. Загрузка данных и вставка
docs = load_documents("./extracted_data.json")
model = SentenceTransformer("cointegrated/rubert-tiny2")

batch_size = 100
total = len(docs)
for i in range(0, total, batch_size):
    batch = docs[i:i+batch_size]
    texts = [d["text"] for d in batch]
    emb = model.encode(texts, normalize_embeddings=True).tolist()
    metas = [d["metadata"] for d in batch]
    ids = [f"doc_{i+j}" for j in range(len(batch))]
    index.upsert(vectors=list(zip(ids, emb, metas)))
    print(f"Добавлено {len(batch)} из {total}")

print(f"✅ Всего добавлено {total} документов.")

# 6. Поиск
query = "налог на прибыль"
qe = model.encode([query], normalize_embeddings=True).tolist()
res = index.query(vector=qe, top_k=3, include_metadata=True)

print("\n🔍 Результаты поиска:")
if res['matches']:
    for match in res['matches']:
        src = match['metadata'].get('source', 'unknown')
        print(f"  - {src}: сходство {match['score']:.4f}")
else:
    print("  Ничего не найдено.")


### 7.6.4. Weaviate – гибридный поиск (BM25 + вектор)




In [ ]:
# ================================================================
# Weaviate – подключение через Colab Secrets (Исправлено для v4)
# ================================================================

!pip install weaviate-client sentence-transformers -q

import json
from sentence_transformers import SentenceTransformer
import weaviate
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import MetadataQuery
from weaviate.classes.init import Auth
from google.colab import userdata

# ---- Загрузка секретов ----
WEAVIATE_URL = userdata.get('WEAVIATE_URL')
WEAVIATE_API_KEY = userdata.get('WEAVIATE_API_KEY')

if not WEAVIATE_URL or not WEAVIATE_API_KEY:
    raise ValueError(
        "❌ Секреты не найдены!\n"
        "Добавьте их в панели 🔑 Secrets:\n"
        "  - WEAVIATE_URL = https://atpl3xhmr8aaimzordfnua.c0.eu-central-1.aws.weaviate.cloud\n"
        "  - WEAVIATE_API_KEY = ваш_секретный_ключ"
    )

# ---- Загрузка документов ----
def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# ---- Подключение ----
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY), # Современный способ аутентификации
)

# ---- Создание коллекции ----
if client.collections.exists("Document"):
    client.collections.delete("Document")

collection = client.collections.create(
    name="Document",
    properties=[
        Property(name="text", data_type=DataType.TEXT),
        Property(name="source", data_type=DataType.TEXT),
        Property(name="author", data_type=DataType.TEXT),
    ],
    vectorizer_config=Configure.Vectorizer.none(), # Явное указание отсутствия встроенного векизатора
)

# ---- Загрузка и вставка ----
docs = load_documents("./extracted_data.json")
model = SentenceTransformer("cointegrated/rubert-tiny2")

with collection.batch.fixed_size(batch_size=100) as batch:
    for doc in docs:
        text = doc["text"]
        emb = model.encode(text, normalize_embeddings=True).tolist()
        properties = {
            "text": text,
            "source": doc["metadata"].get("source", ""),
            "author": doc["metadata"].get("author", ""),
        }
        batch.add_object(properties=properties, vector=emb)

print(f"✅ Добавлено {len(docs)} документов.")

# ---- Векторный поиск ----
query = "налог на прибыль"
# Передаем строку, чтобы получить 1D массив (один вектор), а не список из одного вектора
qe = model.encode(query, normalize_embeddings=True).tolist()

vector_results = collection.query.near_vector(
    near_vector=qe,         # Передается сам вектор
    distance=0.5,           # Отдельный параметр для фильтра по дистанции
    limit=3,
    return_properties=["text", "source"],
    return_metadata=MetadataQuery(distance=True) # Запрашиваем distance для obj.metadata
)

print("\n🔍 Weaviate (векторный поиск):")
for obj in vector_results.objects:
    print(f"  - {obj.properties['source']}: {obj.properties['text'][:150]}... (distance: {obj.metadata.distance:.4f})")

# ---- Гибридный поиск ----
hybrid_results = collection.query.hybrid(
    query=query,
    alpha=0.5,
    limit=3,
    return_properties=["text", "source"],
    return_metadata=MetadataQuery(score=True) # Запрашиваем score для obj.metadata
)

print("\n🔍 Weaviate (гибридный поиск):")
for obj in hybrid_results.objects:
    print(f"  - {obj.properties['source']}: {obj.properties['text'][:150]}... (score: {obj.metadata.score:.4f})")

client.close()


### 7.6.5. Qdrant – гибкая фильтрация payload




In [ ]:
!pip install qdrant-client sentence-transformers

# ================================================================
# Qdrant – локальный режим (без сервера)
# ================================================================

import json
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# ---- Загрузка данных ----
def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# ---- Подключение к локальной БД (хранилище на диске) ----
client = QdrantClient(path="./qdrant_data")  # данные сохранятся в папку qdrant_data

collection_name = "documents"

# ---- Пересоздание коллекции ----
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=312, distance=Distance.COSINE)
)

# ---- Загрузка документов ----
docs = load_documents("./extracted_data.json")
model = SentenceTransformer("cointegrated/rubert-tiny2")

# ---- Подготовка точек ----
points = []
for i, doc in enumerate(docs):
    emb = model.encode(doc["text"], normalize_embeddings=True).tolist()
    points.append(PointStruct(
        id=i,
        vector=emb,
        payload={
            "text": doc["text"],
            "source": doc["metadata"].get("source", ""),
            "author": doc["metadata"].get("author", ""),
        }
    ))

# ---- Вставка данных ----
client.upsert(collection_name=collection_name, points=points)
print(f"✅ Добавлено {len(points)} документов.")

# ---- Поиск ----
query = "налог на прибыль"
qe = model.encode(query, normalize_embeddings=True).tolist()

# Без фильтра (просто поиск)
results = client.search(
    collection_name=collection_name,
    query_vector=qe,
    limit=3,
    with_payload=True,
)

print("\n🔍 Qdrant (поиск без фильтра):")
for hit in results:
    print(f"  - {hit.payload['source']}: {hit.payload['text'][:150]}... (score: {hit.score:.4f})")

# ---- Поиск с фильтром ----
from qdrant_client.models import Filter, FieldCondition, MatchValue

filtered_results = client.search(
    collection_name=collection_name,
    query_vector=qe,
    limit=3,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="source",
                match=MatchValue(value="report_2024.pdf")  # замените на существующий файл
            )
        ]
    ),
    with_payload=True,
)

print("\n🔍 Qdrant (с фильтром по источнику):")
for hit in filtered_results:
    print(f"  - {hit.payload['source']}: {hit.payload['text'][:150]}... (score: {hit.score:.4f})")


### 7.6.6. Milvus – масштабирование до миллиардов




In [ ]:
# ================================================================
# Milvus: распределённый поиск
# ================================================================
!pip install pymilvus sentence-transformers

from pymilvus import connections, Collection, CollectionSchema, FieldSchema, DataType, utility

connections.connect("default", host="localhost", port="19530")
collection_name = "documents"

# Безопасное удаление старой коллекции
if utility.has_collection(collection_name):
    Collection(collection_name).drop()

fields = [
    FieldSchema("id", DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema("text", DataType.VARCHAR, max_length=65535),
    FieldSchema("source", DataType.VARCHAR, max_length=255),
    # Если нужно поле author, раскомментируйте строку ниже:
    # FieldSchema("author", DataType.VARCHAR, max_length=255),
    FieldSchema("embedding", DataType.FLOAT_VECTOR, dim=312),
]
schema = CollectionSchema(fields)
collection = Collection(collection_name, schema)

docs = load_documents()
model = SentenceTransformer("cointegrated/rubert-tiny2")

data = []
for doc in docs:
    emb = model.encode(doc["text"], normalize_embeddings=True).tolist()
    data.append({
        "text": doc["text"],
        "source": doc["metadata"].get("source", ""),
        # "author": doc["metadata"].get("author", ""), # Если добавите поле в схему
        "embedding": emb,
    })

# Вставка данных
collection.insert(data)

# Создание индекса (обязательно для быстрого поиска)
collection.create_index(
    "embedding",
    {"metric_type": "COSINE", "index_type": "IVF_FLAT", "params": {"nlist": 128}}
)

# Загрузка коллекции в память (обязательно перед поиском)
collection.load()

query = "налог на прибыль"
# ИСПРАВЛЕНИЕ: Убираем скобки вокруг query, чтобы получить 1D-массив (один вектор)
qe = model.encode(query, normalize_embeddings=True).tolist()

# ИСПРАВЛЕНИЕ: Передаем [qe], чтобы получился корректный 2D-массив для поиска
results = collection.search(
    [qe],
    "embedding",
    {"metric_type": "COSINE", "params": {"nprobe": 10}},
    limit=3,
    output_fields=["text", "source"] # Сюда же можно добавить "author", если он есть в схеме
)

print("\n🔍 Milvus (векторный поиск):")
# results содержит список объектов Hits (по одному на каждый запрос)
for hits in results:
    for hit in hits:
        # Поля из output_fields доступны через hit.entity
        text = hit.entity.get("text")
        source = hit.entity.get("source")
        print(f"  - {source}: {text[:150]}... (distance: {hit.distance:.4f})")


### 7.6.7. LanceDB – хранение на диске (экономия RAM)




In [ ]:
# ================================================================
# LanceDB: данные на диске, индекс не в памяти
# ================================================================
!pip install lancedb pandas sentence-transformers

# !pip install lancedb pandas sentence-transformers -q

import lancedb
import pandas as pd
from sentence_transformers import SentenceTransformer

db = lancedb.connect("./lancedb_data")
docs = load_documents()
model = SentenceTransformer("cointegrated/rubert-tiny2")

data = []
for doc in docs:
    emb = model.encode(doc["text"], normalize_embeddings=True).tolist()
    data.append({
        "text": doc["text"],
        "source": doc["metadata"].get("source", ""),
        "vector": emb, # LanceDB по умолчанию ищет по полю "vector"
    })

df = pd.DataFrame(data)
table = db.create_table("documents", data=df, mode="overwrite")

# СОЗДАНИЕ ИНДЕКСА (ОПЦИОНАЛЬНО)
# Если у вас БОЛЬШЕ 256 документов, можно создать IVF_PQ индекс для ускорения на больших объемах
if len(docs) >= 256:
    table.create_index(
        metric="cosine",
        index_type="IVF_PQ",
        num_partitions=10,
        num_sub_vectors=16
    )
# Если документов мало, LanceDB сам сделает быстрый flat-поиск без индекса!

query = "налог на прибыль"
# ИСПРАВЛЕНИЕ: Убираем скобки, чтобы получить 1D-массив (один вектор)
qe = model.encode(query, normalize_embeddings=True).tolist()

# ПОИСК
# Явно указываем метрику "cosine", чтобы получить косинусное расстояние
results = table.search(qe).metric("cosine").limit(3).to_pandas()

print("\n🔍 LanceDB (поиск с диска):")
# Метод to_pandas() автоматически добавляет колонку '_distance'
for _, row in results.iterrows():
    print(f"  - {row['source']}: {row['text'][:150]}... (distance: {row['_distance']:.4f})")


### 7.6.8. PgVector – родной SQL и ACID в PostgreSQL




In [ ]:
# ================================================================
# PgVector: векторный поиск внутри PostgreSQL
# ================================================================
!pip install psycopg2-binary sentence-transformers pgvector


# !pip install psycopg2-binary sentence-transformers pgvector -q

import psycopg2
from pgvector.psycopg2 import register_vector # ВАЖНО: импортируем адаптер
from sentence_transformers import SentenceTransformer

# 1. Подключение
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="password",
    host="localhost"
)
# 2. Регистрация типа vector для psycopg2
# Теперь psycopg2 умеет автоматически переводить Python-списки в тип vector
register_vector(conn)
cur = conn.cursor()

# 3. Создание таблицы и расширения
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
cur.execute("DROP TABLE IF EXISTS documents;")
cur.execute("""
    CREATE TABLE documents (
        id SERIAL PRIMARY KEY,
        text TEXT,
        source TEXT,
        embedding vector(312)
    );
""")

docs = load_documents()
model = SentenceTransformer("cointegrated/rubert-tiny2")

# 4. Вставка данных
print(f"📥 Вставка {len(docs)} документов...")
for doc in docs:
    emb = model.encode(doc["text"], normalize_embeddings=True).tolist()
    # Благодаря register_vector, мы можем передавать emb (list) напрямую!
    cur.execute(
        "INSERT INTO documents (text, source, embedding) VALUES (%s, %s, %s)",
        (doc["text"], doc["metadata"].get("source", ""), emb)
    )
conn.commit()

# 5. Создание индекса HNSW для ускорения поиска
# Рекомендуется указывать параметры m и ef_construction для лучших результатов
cur.execute("""
    CREATE INDEX ON documents
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
""")
conn.commit()

query = "налог на прибыль"
# ИСПРАВЛЕНИЕ: Убираем скобки, чтобы получить 1D-массив (один вектор)
qe = model.encode(query, normalize_embeddings=True).tolist()

# 6. Поиск
# Оператор <=> вычисляет косинусное расстояние (cosine distance)
# 1 - (embedding <=> %s) превращает расстояние в сходство (similarity от 0 до 1)
cur.execute("""
    SELECT text, source, 1 - (embedding <=> %s) AS similarity
    FROM documents
    ORDER BY embedding <=> %s
    LIMIT 3;
""", (qe, qe))

results = cur.fetchall()
print("\n🔍 PgVector (SQL поиск):")
for text, source, similarity in results:
    print(f"  - {source}: {text[:150]}... (similarity: {similarity:.4f})")

# 7. ОБЯЗАТЕЛЬНО закрываем ресурсы
cur.close()
conn.close()

In [ ]:
# ================================================================
# УСТАНОВКА БИБЛИОТЕК
# ================================================================

!pip install -q \
    pypdf \
    pdfplumber \
    sentence-transformers \
    chromadb \
    langchain-text-splitters \
    datasets \
    evaluate \
    sacrebleu \
    matplotlib \
    "numpy<2.1"

    # ================================================================
# СКВОЗНОЙ ПРИМЕР: ПОЛНЫЙ RAG-ПАЙПЛАЙН
#
# Объединяет все темы Лекции 5.1:
# - Извлечение текста из PDF (Тема 4)
# - Чанкинг (Тема 5)
# - Эмбеддинги (Тема 6)
# - Векторная БД (Тема 7)
# - Поиск и генерация (Тема 1-2)
# - Оценка качества (Тема 3)
# - Адаптивный RAG (Тема 8)
# ================================================================

import os
import json
import re
import time
import pickle
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import pdfplumber
from pypdf import PdfReader
import evaluate

# --------------------------------------------
# ЧАСТЬ 1: ИЗВЛЕЧЕНИЕ ТЕКСТА (Тема 4)
# --------------------------------------------

def extract_pdf_text(file_path: str) -> str:
    """
    Извлекает текст из PDF с fallback между pdfplumber и pypdf.
    """
    text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        if text.strip():
            return text
    except Exception as e:
        print(f"pdfplumber не сработал для {file_path}: {e}")

    try:
        with open(file_path, "rb") as f:
            reader = PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"Не удалось извлечь текст из {file_path}: {e}")

    return text.strip()


def clean_text(text: str) -> str:
    """Очищает текст от шума."""
    if not text:
        return ""
    text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def load_documents_from_folder(folder_path: str = "./documents") -> List[Dict]:
    """
    Загружает все PDF из папки и возвращает список документов с метаданными.
    """
    results = []
    pdf_files = list(Path(folder_path).glob("**/*.pdf"))

    if not pdf_files:
        print(f"⚠️ PDF файлы не найдены в {folder_path}")
        print("Создаю демонстрационные документы...")
        return create_demo_documents()

    print(f"📁 Найдено {len(pdf_files)} PDF файлов")

    for pdf_path in pdf_files:
        text = extract_pdf_text(str(pdf_path))
        if not text:
            continue
        cleaned_text = clean_text(text)
        results.append({
            "text": cleaned_text,
            "metadata": {
                "source": pdf_path.name,
                "file_path": str(pdf_path),
                "doc_id": pdf_path.stem,
            }
        })
        print(f"  ✅ Загружен: {pdf_path.name} ({len(cleaned_text)} символов)")

    return results


def create_demo_documents() -> List[Dict]:
    """
    Создаёт демонстрационные документы для примера.
    """
    docs = [
        {
            "text": """
            Налог на прибыль организаций регулируется главой 25 Налогового кодекса РФ.
            Ставка налога составляет 20% от прибыли. При этом 3% зачисляется в федеральный бюджет,
            17% — в региональный бюджет. Для IT-компаний предусмотрена льготная ставка 17%.
            Льгота действует при условии, что доля IT-доходов составляет не менее 70%.
            """,
            "metadata": {"source": "Налоговый кодекс РФ", "doc_id": "tax_code"}
        },
        {
            "text": """
            Самозанятые граждане уплачивают налог на профессиональный доход (НПД).
            Ставка налога составляет 4% при работе с физическими лицами и 6% при работе
            с юридическими лицами. Максимальный годовой доход для применения НПД — 2.4 млн рублей.
            Налог уплачивается ежемесячно до 25 числа следующего месяца.
            """,
            "metadata": {"source": "Закон о НПД", "doc_id": "npd_law"}
        },
        {
            "text": """
            Индивидуальные предприниматели обязаны вести книгу учёта доходов и расходов (КУДиР).
            Отчётность сдаётся в налоговую по месту регистрации. Срок сдачи декларации —
            30 апреля следующего года. УСН позволяет уплачивать налог по ставке 6% от доходов
            или 15% от доходов минус расходы.
            """,
            "metadata": {"source": "Инструкция для ИП", "doc_id": "ip_guide"}
        },
        {
            "text": """
            Страховые взносы во внебюджетные фонды уплачиваются всеми работодателями.
            В 2024 году ставка составляет 30% от фонда оплаты труда. Пенсионный фонд — 22%,
            Фонд социального страхования — 2.9%, Фонд обязательного медицинского страхования — 5.1%.
            Для малого бизнеса предусмотрена пониженная ставка 15% на сумму превышения МРОТ.
            """,
            "metadata": {"source": "Страховые взносы", "doc_id": "insurance"}
        }
    ]
    return docs


# --------------------------------------------
# ЧАСТЬ 2: ЧАНКИНГ (Тема 5)
# --------------------------------------------

class RecursiveTextSplitter:
    """Рекурсивный сплиттер для разбиения текста на чанки."""

    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.separators = ["\n\n", "\n", ". ", "! ", "? ", ", ", " "]

    def split_document(self, text: str, metadata: Dict) -> List[Dict]:
        chunks_text = self._split_text(text)
        chunks_with_meta = []
        for i, chunk_text in enumerate(chunks_text):
            chunk_meta = metadata.copy()
            chunk_meta["chunk_index"] = i
            chunk_meta["chunk_length"] = len(chunk_text)
            chunks_with_meta.append({"text": chunk_text, "metadata": chunk_meta})
        return chunks_with_meta

    def _split_text(self, text: str) -> List[str]:
        if not text:
            return []
        chunks = []
        current_chunk = []
        current_len = 0
        segments = self._split_by_separators(text, self.separators)

        for segment in segments:
            seg_len = len(segment)
            if current_len + seg_len > self.chunk_size and current_chunk:
                chunks.append("".join(current_chunk).strip())
                overlap_text = self._get_overlap("".join(current_chunk), self.chunk_overlap)
                current_chunk = [overlap_text]
                current_len = len(overlap_text)
            current_chunk.append(segment)
            current_len += seg_len

        if current_chunk:
            chunks.append("".join(current_chunk).strip())
        return chunks

    def _split_by_separators(self, text: str, separators: List[str]) -> List[str]:
        if not text:
            return []
        separator = separators[0]
        remaining_seps = separators[1:]
        if not remaining_seps:
            return text.split(separator)
        parts = text.split(separator)
        result = []
        for i, part in enumerate(parts):
            if len(part) <= self.chunk_size:
                result.append(part)
            else:
                sub_parts = self._split_by_separators(part, remaining_seps)
                result.extend(sub_parts)
            if i < len(parts) - 1:
                result.append(separator)
        return result

    def _get_overlap(self, text: str, overlap_len: int) -> str:
        return text[-overlap_len:] if len(text) > overlap_len else text


def process_chunking(documents: List[Dict],
                     chunk_size: int = 500,
                     chunk_overlap: int = 50) -> List[Dict]:
    """Применяет чанкинг ко всем документам."""
    splitter = RecursiveTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    all_chunks = []
    for doc in documents:
        chunks = splitter.split_document(doc["text"], doc["metadata"])
        all_chunks.extend(chunks)
    print(f"📊 Создано {len(all_chunks)} чанков (размер={chunk_size}, overlap={chunk_overlap})")
    return all_chunks


# --------------------------------------------
# ЧАСТЬ 3: ЭМБЕДДИНГИ (Тема 6)
# --------------------------------------------

class EmbeddingGenerator:
    """Генератор эмбеддингов с автоматическим определением устройства."""

    def __init__(self, model_name: str = "cointegrated/rubert-tiny2", device: str = None):
        import torch
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name
        self.dimension = self.model.get_sentence_embedding_dimension()
        print(f"🧠 Модель: {model_name}, размерность: {self.dimension}, устройство: {device}")

    def encode_batch(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        return self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )


def generate_embeddings_for_chunks(chunks: List[Dict],
                                   model_name: str = "cointegrated/rubert-tiny2") -> np.ndarray:
    """Генерирует эмбеддинги для всех чанков."""
    generator = EmbeddingGenerator(model_name)
    texts = [chunk["text"] for chunk in chunks]
    embeddings = generator.encode_batch(texts, batch_size=32)
    print(f"✅ Сгенерировано {len(embeddings)} эмбеддингов размерности {embeddings.shape[1]}")
    return embeddings


# --------------------------------------------
# ЧАСТЬ 4: ВЕКТОРНАЯ БД (Тема 7)
# --------------------------------------------

def build_chroma_db(chunks: List[Dict], embeddings: np.ndarray, persist_dir: str = "./chroma_db") -> chromadb.Collection:
    """Строит векторную БД в Chroma."""
    client = chromadb.PersistentClient(
        path=persist_dir,
        settings=Settings(anonymized_telemetry=False)
    )

    # Удаляем старую коллекцию
    try:
        client.delete_collection("documents")
    except:
        pass

    collection = client.create_collection(
        name="documents",
        metadata={"hnsw:space": "cosine"}
    )

    texts = [chunk["text"] for chunk in chunks]
    metadatas = [chunk["metadata"] for chunk in chunks]
    ids = [f"chunk_{i:04d}" for i in range(len(chunks))]

    collection.add(
        documents=texts,
        embeddings=embeddings.tolist(),
        metadatas=metadatas,
        ids=ids
    )

    print(f"✅ Векторная БД построена: {len(chunks)} векторов")
    return collection


# --------------------------------------------
# ЧАСТЬ 5: ПОИСК И ГЕНЕРАЦИЯ (Тема 1-2)
# --------------------------------------------

def search_chunks(collection: chromadb.Collection,
                  query: str,
                  model: SentenceTransformer,
                  n_results: int = 3) -> List[Dict]:
    """Выполняет поиск в векторной БД."""
    query_emb = model.encode([query], normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=query_emb,
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    top_chunks = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        top_chunks.append({
            "text": doc,
            "metadata": meta,
            "distance": dist
        })

    return top_chunks


def generate_response(query: str,
                      top_chunks: List[Dict],
                      use_llm: bool = True,
                      llm_func=None) -> str:
    """
    Генерирует ответ на основе найденных чанков.
    Если use_llm=False — возвращает простой форматированный ответ.
    """
    if not top_chunks:
        return "К сожалению, не удалось найти релевантную информацию."

    # Формируем контекст из чанков
    context = "\n\n".join([
        f"[Источник: {chunk['metadata'].get('source', 'unknown')}]\n{chunk['text']}"
        for chunk in top_chunks
    ])

    if use_llm and llm_func:
        # Используем LLM для генерации ответа
        prompt = f"""
        Ты — корпоративный ассистент. Ответь на вопрос, используя только предоставленный контекст.
        Если в контексте нет информации для ответа, скажи об этом честно.
        Укажи источники информации.

        Контекст:
        {context}

        Вопрос: {query}

        Ответ:
        """
        return llm_func(prompt)
    else:
        # Простой форматированный ответ (без LLM)
        response = f"**Вопрос:** {query}\n\n"
        response += f"**Найдено {len(top_chunks)} источников:**\n\n"
        for i, chunk in enumerate(top_chunks, 1):
            source = chunk['metadata'].get('source', 'unknown')
            response += f"**{i}. Источник: {source}**\n"
            response += f"{chunk['text'][:300]}...\n"
            response += f"*(релевантность: {1 - chunk['distance']:.3f})*\n\n"
        return response


# --------------------------------------------
# ЧАСТЬ 6: ОЦЕНКА КАЧЕСТВА (Тема 3)
# --------------------------------------------

def evaluate_search_quality(collection: chromadb.Collection,
                           model: SentenceTransformer,
                           test_queries: List[Dict]) -> Dict:
    """
    Оценивает качество поиска на тестовых запросах с известными ответами.
    """
    bleu = evaluate.load("sacrebleu")

    results = []

    for test in test_queries:
        query = test["query"]
        expected_keywords = test.get("keywords", [])

        top_chunks = search_chunks(collection, query, model, n_results=3)

        # Проверяем, содержатся ли ожидаемые ключевые слова в найденных чанках
        found_keywords = []
        for chunk in top_chunks:
            text = chunk["text"].lower()
            for kw in expected_keywords:
                if kw.lower() in text and kw not in found_keywords:
                    found_keywords.append(kw)

        recall = len(found_keywords) / len(expected_keywords) if expected_keywords else 1.0

        results.append({
            "query": query,
            "found_keywords": found_keywords,
            "expected_keywords": expected_keywords,
            "recall": recall,
            "top_sources": [chunk["metadata"].get("source", "unknown") for chunk in top_chunks],
            "top_distances": [chunk["distance"] for chunk in top_chunks],
        })

    avg_recall = np.mean([r["recall"] for r in results])

    print("\n" + "="*60)
    print("📊 ОЦЕНКА КАЧЕСТВА ПОИСКА")
    print("="*60)
    for r in results:
        status = "✅" if r["recall"] == 1.0 else "⚠️"
        print(f"{status} '{r['query']}': recall={r['recall']:.2%}, "
              f"найдено={r['found_keywords']}, ожидалось={r['expected_keywords']}")
    print(f"\nСредний Recall: {avg_recall:.2%}")
    print("="*60)

    return {"results": results, "avg_recall": avg_recall}


# --------------------------------------------
# ЧАСТЬ 7: АДАПТИВНЫЙ RAG (Тема 8)
# --------------------------------------------

class AdaptiveRAG:
    """
    Адаптивный RAG с классификацией сложности запроса.
    """

    def __init__(self,
                 collection: chromadb.Collection,
                 model: SentenceTransformer,
                 llm_func=None,
                 complexity_threshold: int = 30):
        self.collection = collection
        self.model = model
        self.llm_func = llm_func
        self.complexity_threshold = complexity_threshold

    def classify_query(self, query: str) -> str:
        """
        Классифицирует запрос по сложности.
        - 'simple': отвечаем без поиска (короткие, общие вопросы)
        - 'complex': используем полный RAG
        """
        # Эвристика: длина запроса + ключевые слова сложности
        words = query.split()
        word_count = len(words)

        complexity_keywords = ['сравни', 'отличие', 'анализ', 'детально', 'подробно',
                              'почему', 'каким образом', 'влияет', 'зависит']

        has_complex_keywords = any(kw in query.lower() for kw in complexity_keywords)

        if word_count < self.complexity_threshold and not has_complex_keywords:
            return "simple"
        else:
            return "complex"

    def answer(self, query: str) -> str:
        """Обрабатывает запрос с учётом классификации сложности."""
        complexity = self.classify_query(query)

        if complexity == "simple":
            # Простой вопрос: отвечаем без поиска (только LLM)
            print(f"🔹 Простой запрос (без поиска): '{query}'")
            if self.llm_func:
                prompt = f"Ты — корпоративный ассистент. Ответь кратко на вопрос: {query}"
                return self.llm_func(prompt)
            else:
                return f"❓ Простой вопрос: {query}\n(Для полного ответа используйте LLM)"
        else:
            # Сложный вопрос: используем полный RAG
            print(f"🔸 Сложный запрос (с поиском): '{query}'")
            top_chunks = search_chunks(self.collection, query, self.model, n_results=3)
            return generate_response(query, top_chunks, use_llm=bool(self.llm_func), llm_func=self.llm_func)


# --------------------------------------------
# ЧАСТЬ 8: ВИЗУАЛИЗАЦИЯ
# --------------------------------------------

def plot_results(results: Dict, save_path: str = "./rag_results_plot.png"):
    """Визуализирует результаты оценки."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # График 1: Recall по запросам
    recalls = [r["recall"] for r in results["results"]]
    queries = [r["query"][:30] + "..." if len(r["query"]) > 30 else r["query"] for r in results["results"]]

    axes[0].bar(queries, recalls, color=['green' if r == 1 else 'orange' for r in recalls])
    axes[0].set_ylabel('Recall')
    axes[0].set_title('Качество поиска по запросам')
    axes[0].set_ylim(0, 1.1)
    axes[0].axhline(y=1.0, color='green', linestyle='--', label='Идеальный recall')
    axes[0].legend()
    axes[0].tick_params(axis='x', rotation=45)

    # График 2: Распределение расстояний
    all_distances = []
    for r in results["results"]:
        all_distances.extend(r["top_distances"])

    axes[1].hist(all_distances, bins=20, alpha=0.7, color='blue', edgecolor='black')
    axes[1].set_xlabel('Косинусное расстояние (0 = близко)')
    axes[1].set_ylabel('Частота')
    axes[1].set_title('Распределение расстояний до найденных чанков')
    axes[1].axvline(x=np.mean(all_distances), color='red', linestyle='--', label=f'Среднее: {np.mean(all_distances):.3f}')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"📊 График сохранён: {save_path}")


# --------------------------------------------
# ГЛАВНАЯ ФУНКЦИЯ
# --------------------------------------------

def main():
    """Запускает полный RAG-пайплайн."""
    print("\n" + "="*70)
    print("🚀 ЗАПУСК ПОЛНОГО RAG-ПАЙПЛАЙНА")
    print("="*70)

    # === ШАГ 1: Загрузка документов (Тема 4) ===
    print("\n📂 ШАГ 1: ЗАГРУЗКА ДОКУМЕНТОВ")
    print("-"*50)
    documents = load_documents_from_folder("./documents")
    print(f"✅ Загружено {len(documents)} документов")

    # === ШАГ 2: Чанкинг (Тема 5) ===
    print("\n✂️ ШАГ 2: ЧАНКИНГ")
    print("-"*50)
    chunks = process_chunking(documents, chunk_size=500, chunk_overlap=50)

    # === ШАГ 3: Эмбеддинги (Тема 6) ===
    print("\n🧠 ШАГ 3: ГЕНЕРАЦИЯ ЭМБЕДДИНГОВ")
    print("-"*50)
    model_name = "cointegrated/rubert-tiny2"
    embeddings = generate_embeddings_for_chunks(chunks, model_name)

    # === ШАГ 4: Векторная БД (Тема 7) ===
    print("\n🗄️ ШАГ 4: ПОСТРОЕНИЕ ВЕКТОРНОЙ БД")
    print("-"*50)
    collection = build_chroma_db(chunks, embeddings, "./chroma_db")

    # === ШАГ 5: Поиск и генерация (Тема 1-2) ===
    print("\n🔍 ШАГ 5: ПОИСК И ГЕНЕРАЦИЯ")
    print("-"*50)

    model = SentenceTransformer(model_name)

    test_queries = [
        "Какой налог платят самозанятые?",
        "Какая ставка налога на прибыль для IT-компаний?",
        "Что такое КУДиР и кто его должен вести?",
        "Какой процент страховых взносов платят работодатели?",
    ]

    print("\nРезультаты поиска:")
    for query in test_queries:
        print(f"\n📌 Вопрос: {query}")
        top_chunks = search_chunks(collection, query, model, n_results=3)

        for i, chunk in enumerate(top_chunks, 1):
            source = chunk["metadata"].get("source", "unknown")
            score = 1 - chunk["distance"]
            print(f"  {i}. [{source}] {chunk['text'][:120]}... (score={score:.3f})")

    # === ШАГ 6: Оценка качества (Тема 3) ===
    print("\n📊 ШАГ 6: ОЦЕНКА КАЧЕСТВА")
    print("-"*50)

    eval_queries = [
        {"query": "Какой налог платят самозанятые?", "keywords": ["НПД", "4%", "6%"]},
        {"query": "Ставка налога на прибыль", "keywords": ["20%", "IT-компаний", "17%"]},
        {"query": "Страховые взносы работодателей", "keywords": ["30%", "Пенсионный", "ФОТ"]},
    ]

    eval_results = evaluate_search_quality(collection, model, eval_queries)

    # === ШАГ 7: Адаптивный RAG (Тема 8) ===
    print("\n🔄 ШАГ 7: АДАПТИВНЫЙ RAG")
    print("-"*50)

    # Простая функция-заглушка для LLM (в реальности замените на Ollama или OpenAI)
    def dummy_llm(prompt: str) -> str:
        return f"🤖 (LLM ответ на: {prompt[:50]}...)"

    adaptive_rag = AdaptiveRAG(
        collection=collection,
        model=model,
        llm_func=dummy_llm,
        complexity_threshold=20
    )

    mixed_queries = [
        "Здравствуйте",
        "Какой налог платят самозанятые?",
        "Сравните налогообложение для самозанятых и ИП",
    ]

    print("\nТестирование адаптивного RAG:")
    for query in mixed_queries:
        print(f"\n📌 Запрос: '{query}'")
        response = adaptive_rag.answer(query)
        print(f"Ответ: {response[:200]}...")

    # === ШАГ 8: Визуализация ===
    print("\n📈 ШАГ 8: ВИЗУАЛИЗАЦИЯ")
    print("-"*50)
    plot_results(eval_results, "./rag_results_plot.png")

    # === ИТОГИ ===
    print("\n" + "="*70)
    print("✅ RAG-ПАЙПЛАЙН УСПЕШНО ЗАВЕРШЁН")
    print("="*70)
    print(f"📄 Документов: {len(documents)}")
    print(f"✂️ Чанков: {len(chunks)}")
    print(f"🧠 Размерность эмбеддингов: {embeddings.shape[1]}")
    print(f"🗄️ Векторная БД: Chroma ({len(collection.get()['ids'])} векторов)")
    print(f"📊 Средний Recall: {eval_results['avg_recall']:.2%}")
    print("="*70)


# --------------------------------------------
# ЗАПУСК
# --------------------------------------------

if __name__ == "__main__":
    main()

# Лекция 5.2 – Интеграция RAG с LLM и оптимизация

## Тема 1. Подключение LLM к RAG-системе (расширенный)

In [ ]:
# ================================================================
# RAG-система с локальными LLM через Ollama
# Исправленная версия с подавлением предупреждений и GPU
# ================================================================

!pip install -q requests sentence-transformers chromadb torch

import os
import sys
import time
import json
import requests
import subprocess
import warnings
import torch
from typing import Optional, List, Dict
import chromadb
from sentence_transformers import SentenceTransformer

# Подавление предупреждения UNEXPECTED (не влияет на работу)
warnings.filterwarnings("ignore", message=".*UNEXPECTED.*")

# ========== 1. ОПРЕДЕЛЕНИЕ СРЕДЫ ==========
IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
print(f"🌐 Среда: {'Google Colab' if IS_COLAB else 'Локально'}")

# ========== 2. НАСТРОЙКА ХРАНИЛИЩА ==========
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DB_PATH = "/content/drive/MyDrive/chroma_db"
    print("✅ Google Drive смонтирован, БД будет сохранена в MyDrive")
else:
    DB_PATH = "./chroma_db"

# ========== 3. ВЫБОР МОДЕЛИ ==========
MODEL_NAME = "qwen2.5:1.5b"  # Можно заменить на "deepseek-r1:1.5b"
print(f"🤖 Используемая модель: {MODEL_NAME}")

# ========== 4. УСТАНОВКА И ЗАПУСК OLLAMA ==========
def ensure_model(model_name):
    """Проверяет, загружена ли модель, и скачивает при необходимости."""
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        if r.status_code == 200:
            models = [m['name'] for m in r.json().get('models', [])]
            if model_name in models:
                print(f"✅ Модель {model_name} уже загружена")
                return True
            else:
                print(f"📥 Модель {model_name} не найдена, скачиваем...")
                result = subprocess.run(["ollama", "pull", model_name], capture_output=True, text=True)
                if result.returncode == 0:
                    print(f"✅ Модель {model_name} успешно загружена")
                    return True
                else:
                    print(f"❌ Ошибка при загрузке: {result.stderr}")
                    return False
    except Exception as e:
        print(f"⚠️ Не удалось проверить модели: {e}")
        return False

def setup_ollama():
    if not IS_COLAB:
        try:
            subprocess.run(["ollama", "--version"], check=True, capture_output=True)
            print("✅ Ollama уже установлен локально")
            return ensure_model(MODEL_NAME)
        except:
            print("⚠️ Ollama не найден, установите вручную")
            return False

    # Проверяем, запущен ли сервер
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("✅ Ollama уже запущен")
        return ensure_model(MODEL_NAME)
    except:
        pass

    print("📦 Установка Ollama в Colab...")
    !apt-get install -y zstd pciutils > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh

    os.environ["PATH"] += os.pathsep + "/usr/local/bin"
    try:
        result = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
        print(f"  ✅ Ollama установлен: {result.stdout.strip()}")
    except FileNotFoundError:
        print("  ❌ Ошибка установки")
        return False

    print("  → Запуск сервера...")
    os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
    proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True
    )
    print(f"  ✅ Сервер запущен (PID: {proc.pid})")

    for _ in range(30):
        try:
            requests.get("http://localhost:11434/api/tags", timeout=2)
            print("  ✅ Ollama готов!")
            break
        except:
            time.sleep(1)
    else:
        print("  ❌ Не дождались сервера")
        return False

    return ensure_model(MODEL_NAME)

OLLAMA_AVAILABLE = setup_ollama()
if not OLLAMA_AVAILABLE:
    print("❌ Не удалось запустить Ollama или загрузить модель.")
    sys.exit(1)

# ========== 5. ОБРАЩЕНИЕ К OLLAMA С ПОВТОРАМИ ==========
def query_ollama_with_retry(prompt, model=MODEL_NAME, temperature=0.7, max_tokens=512,
                            system=None, retries=3, timeout=300):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": max_tokens}
    }
    if system:
        payload["system"] = system

    for attempt in range(retries):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json().get("response", "")
        except requests.exceptions.Timeout:
            print(f"  ⏳ Таймаут (попытка {attempt+1}/{retries}), повтор через {2**attempt} сек...")
            time.sleep(2 ** attempt)
        except Exception as e:
            print(f"  ❌ Ошибка: {e} (попытка {attempt+1}/{retries})")
            time.sleep(2 ** attempt)
    return "[Ошибка] Не удалось получить ответ после нескольких попыток."

class OllamaAdapter:
    def __init__(self, model=MODEL_NAME):
        self.model = model
    def generate(self, prompt, system=None, temperature=0.7, max_tokens=512):
        return query_ollama_with_retry(prompt, self.model, temperature, max_tokens, system)

# ========== 6. RAG СИСТЕМА ==========
class RAGSystem:
    def __init__(self, llm_adapter, embed_model_name="cointegrated/rubert-tiny2", db_path=DB_PATH):
        self.llm = llm_adapter
        # Используем GPU, если доступен
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"🔧 Устройство для эмбеддингов: {device}")
        self.embed_model = SentenceTransformer(embed_model_name, device=device)
        self.db_path = db_path
        self.client = chromadb.PersistentClient(path=db_path)
        self.collection = None
        self._cache = {}
        self._init_collection()

    def _init_collection(self):
        try:
            self.collection = self.client.get_collection("documents")
            print(f"ℹ️ База 'documents' загружена, документов: {self.collection.count()}")
        except:
            self.collection = self.client.create_collection("documents")
            print("🆕 Создана новая коллекция 'documents'")

    def add_documents(self, documents: List[str], metadatas: List[Dict] = None):
        if metadatas is None:
            metadatas = [{} for _ in documents]

        embeddings = []
        for doc in documents:
            if doc not in self._cache:
                emb = self.embed_model.encode([doc], normalize_embeddings=True).tolist()[0]
                self._cache[doc] = emb
            embeddings.append(self._cache[doc])

        ids = [f"doc_{i}_{int(time.time())}" for i in range(len(documents))]
        self.collection.add(
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas,
            ids=ids
        )
        print(f"✅ Добавлено {len(documents)} документов")

    def query(self, question: str, n_results: int = 3, system_prompt: str = None,
              max_context_tokens: int = 2000) -> str:
        q_emb = self.embed_model.encode([question], normalize_embeddings=True).tolist()
        results = self.collection.query(query_embeddings=q_emb, n_results=n_results)

        if not results['documents'] or not results['documents'][0]:
            return "❌ Нет релевантных документов."

        context = ""
        for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
            source = meta.get('source', 'неизвестно')
            context += f"\n[Документ {i+1}] Источник: {source}\n{doc}\n"

        # Усечение контекста, если он слишком длинный
        if len(context) // 4 > max_context_tokens:
            context = context[:max_context_tokens * 4] + "\n...[контекст обрезан]"

        default_system = ("Ты — юридический консультант. Отвечай строго по контексту. "
                          "Не добавляй свои знания. Если ответа нет в контексте, скажи: 'В контексте нет информации'.")
        system = system_prompt or default_system

        prompt = f"Контекст:\n{context}\n\nВопрос: {question}\nОтвет:"
        return self.llm.generate(prompt, system=system, temperature=0.3, max_tokens=512)

# ========== 7. ДЕМОНСТРАЦИЯ ==========
def demo():
    print("=" * 60)
    print("🧪 Проверка Ollama и модели")
    print("=" * 60)

    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=3)
        models = [m['name'] for m in r.json().get('models', [])]
        print(f"✅ Ollama доступен. Загруженные модели: {', '.join(models)}")
        if MODEL_NAME not in models:
            print(f"⚠️ Модель {MODEL_NAME} не найдена, попытка загрузить...")
            ensure_model(MODEL_NAME)
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        return

    llm = OllamaAdapter(MODEL_NAME)
    rag = RAGSystem(llm)

    if rag.collection.count() == 0:
        print("🛠️ Наполняем базу тестовыми документами...")
        docs = [
            "Самозанятые граждане платят налог на профессиональный доход (НПД). Ставка 4% при работе с физлицами и 6% с юрлицами.",
            "Для российских IT-компаний с аккредитацией Минцифры налог на прибыль 0% до конца 2024, с 2025 — 5%.",
            "Страховые взносы для самозанятых необязательны, можно платить добровольно для пенсии.",
            "IT-компании освобождены от НДС при продаже собственного ПО."
        ]
        metas = [
            {"source": "Закон о НПД (ФЗ-422)"},
            {"source": "НК РФ (Ст. 284.5)"},
            {"source": "Закон о НПД (Ст. 14)"},
            {"source": "НК РФ (Ст. 145.1)"}
        ]
        rag.add_documents(docs, metas)

    questions = [
        "Какой налог платят самозанятые?",
        "Ставка налога на прибыль для IT-компаний?",
        "Обязательны ли страховые взносы для самозанятых?"
    ]

    print("\n" + "=" * 60)
    print("🔎 RAG-запросы")
    print("=" * 60)
    for q in questions:
        print(f"\n📌 Вопрос: {q}")
        answer = rag.query(q)
        print(f"   Ответ: {answer[:600]}{'...' if len(answer)>600 else ''}")

if __name__ == "__main__":
    demo()

# Тема 2. Расширенный поиск и переранжирование (расширенный)

In [ ]:
# ================================================================
# Тема 2. Расширенный поиск и переранжирование
# Полный код для Google Colab (локальный запуск)
# ================================================================

!pip install -q requests sentence-transformers chromadb rank-bm25

import os
import sys
import time
import json
import requests
import subprocess
import numpy as np
from typing import List, Dict, Tuple, Optional
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
print(f"Среда: {'Colab' if IS_COLAB else 'локально'}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DB_PATH = "/content/drive/MyDrive/chroma_db"
else:
    DB_PATH = "./chroma_db"

MODEL_NAME = "qwen2.5:1.5b"

def ensure_model(model_name):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        if r.status_code == 200:
            models = [m['name'] for m in r.json().get('models', [])]
            if model_name in models:
                print(f"Модель {model_name} уже загружена")
                return True
            else:
                print(f"Загрузка {model_name}...")
                subprocess.run(["ollama", "pull", model_name], check=True, capture_output=True)
                print(f"Модель {model_name} загружена")
                return True
    except Exception as e:
        print(f"Ошибка: {e}")
        return False

def setup_ollama():
    if not IS_COLAB:
        try:
            subprocess.run(["ollama", "--version"], check=True, capture_output=True)
            return ensure_model(MODEL_NAME)
        except:
            print("Ollama не найден, установите вручную")
            return False
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("Ollama уже запущен")
        return ensure_model(MODEL_NAME)
    except:
        pass

    print("Установка Ollama...")
    !apt-get install -y zstd pciutils > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh
    os.environ["PATH"] += os.pathsep + "/usr/local/bin"
    try:
        subprocess.run(["ollama", "--version"], check=True, capture_output=True)
    except:
        print("Ошибка установки")
        return False

    print("Запуск сервера...")
    os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(30):
        try:
            requests.get("http://localhost:11434/api/tags", timeout=2)
            print("Ollama готов")
            break
        except:
            time.sleep(1)
    else:
        print("Сервер не запустился")
        return False
    return ensure_model(MODEL_NAME)

if not setup_ollama():
    sys.exit(1)

def query_ollama(prompt, model=MODEL_NAME, temperature=0.7, max_tokens=512, system=None, retries=3, timeout=300):
    url = "http://localhost:11434/api/generate"
    payload = {"model": model, "prompt": prompt, "stream": False, "options": {"temperature": temperature, "num_predict": max_tokens}}
    if system:
        payload["system"] = system
    for attempt in range(retries):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json().get("response", "")
        except Exception as e:
            print(f"Попытка {attempt+1} ошибка: {e}")
            time.sleep(2 ** attempt)
    return "[Ошибка]"

class OllamaAdapter:
    def __init__(self, model=MODEL_NAME):
        self.model = model
    def generate(self, prompt, system=None, temperature=0.7, max_tokens=512):
        return query_ollama(prompt, self.model, temperature, max_tokens, system)

class AdvancedRetriever:
    def __init__(self, collection, embed_model, texts, metadatas, alpha=0.6, k1=1.2, b=0.75,
                 rerank_top_k=20, final_top_k=5):
        self.collection = collection
        self.embed_model = embed_model
        self.texts = texts
        self.metadatas = metadatas
        self.alpha = alpha
        self.rerank_top_k = rerank_top_k
        self.final_top_k = final_top_k
        tokenized = [doc.split() for doc in texts]
        self.bm25 = BM25Okapi(tokenized, k1=k1, b=b)
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        print("Cross-encoder загружен")

    def _vector_search(self, query, top_k):
        q_emb = self.embed_model.encode([query], normalize_embeddings=True).tolist()
        res = self.collection.query(query_embeddings=q_emb, n_results=top_k)
        ids = res['ids'][0]
        dist = res['distances'][0]
        sim = [1 - d for d in dist]
        return ids, sim

    def _bm25_search(self, query, top_k):
        tok = query.split()
        scores = self.bm25.get_scores(tok)
        indices = np.argsort(scores)[::-1][:top_k]
        return indices, scores

    def search(self, query, filter_metadata=None):
        vec_ids, vec_scores = self._vector_search(query, self.rerank_top_k * 2)
        bm25_indices, all_scores = self._bm25_search(query, self.rerank_top_k * 2)
        max_bm25 = max(all_scores) if all_scores.any() else 1.0

        combined = {}
        for idx, score in zip(vec_ids, vec_scores):
            doc_idx = int(idx.split('_')[1])
            combined[doc_idx] = self.alpha * score
        for doc_idx in bm25_indices:
            score = all_scores[doc_idx] / max_bm25 if max_bm25 > 0 else 0
            combined[doc_idx] = combined.get(doc_idx, 0) + (1 - self.alpha) * score

        sorted_items = sorted(combined.items(), key=lambda x: x[1], reverse=True)

        # дедупликация по тексту
        seen = set()
        unique = []
        for doc_idx, score in sorted_items:
            text = self.texts[doc_idx]
            if text not in seen:
                seen.add(text)
                unique.append((doc_idx, score))
        sorted_items = unique

        if filter_metadata:
            filtered = []
            for doc_idx, score in sorted_items:
                meta = self.metadatas[doc_idx]
                if all(meta.get(k) == v for k, v in filter_metadata.items()):
                    filtered.append((doc_idx, score))
            sorted_items = filtered

        candidates = sorted_items[:self.rerank_top_k]
        if not candidates:
            return []

        candidate_texts = [self.texts[idx] for idx, _ in candidates]
        pairs = [[query, doc] for doc in candidate_texts]
        rerank_scores = self.reranker.predict(pairs)

        final_indices = np.argsort(rerank_scores)[::-1][:self.final_top_k]
        results = []
        for i in final_indices:
            doc_idx = candidates[i][0]
            results.append({
                'doc_idx': doc_idx,
                'text': self.texts[doc_idx],
                'metadata': self.metadatas[doc_idx],
                'score': float(rerank_scores[i])
            })
        return results

class RAGSystemAdvanced:
    def __init__(self, llm_adapter, embed_model, texts, metadatas, db_path=DB_PATH):
        self.llm = llm_adapter
        self.embed_model = embed_model
        self.texts = texts
        self.metadatas = metadatas
        self.client = chromadb.PersistentClient(path=db_path)
        self.collection = None
        self._init_collection()
        self.retriever = AdvancedRetriever(
            self.collection, self.embed_model, texts, metadatas,
            alpha=0.6, rerank_top_k=20, final_top_k=5
        )

    def _init_collection(self):
        try:
            self.collection = self.client.get_collection("documents")
            print(f"Загружено документов: {self.collection.count()}")
        except:
            self.collection = self.client.create_collection("documents")
            print("Создана новая коллекция")

    def add_documents(self, documents, metadatas=None):
        if metadatas is None:
            metadatas = [{} for _ in documents]
        embeddings = self.embed_model.encode(documents, normalize_embeddings=True).tolist()
        ids = [f"doc_{i}" for i in range(len(documents))]
        self.collection.add(documents=documents, embeddings=embeddings, metadatas=metadatas, ids=ids)
        print(f"Добавлено {len(documents)} документов")

    def query(self, question, filter_metadata=None, system_prompt=None):
        retrieved = self.retriever.search(question, filter_metadata=filter_metadata)
        if not retrieved:
            return "Нет релевантных документов."
        context = ""
        for i, doc in enumerate(retrieved):
            src = doc['metadata'].get('source', 'неизвестно')
            context += f"\n[Документ {i+1}] Источник: {src}\n{doc['text']}\n"
        default_system = "Ты — юридический консультант. Отвечай строго по контексту. Если ответа нет в контексте, скажи об этом."
        system = system_prompt or default_system
        prompt = f"Контекст:\n{context}\n\nВопрос: {question}\nОтвет:"
        return self.llm.generate(prompt, system=system, temperature=0.3, max_tokens=512)

    def search_only(self, question, filter_metadata=None):
        return self.retriever.search(question, filter_metadata=filter_metadata)

def compute_recall_at_k(retrieved_ids, relevant_ids, k=5):
    if not relevant_ids:
        return 0.0
    retrieved_set = set(retrieved_ids[:k])
    relevant_set = set(relevant_ids)
    return len(retrieved_set & relevant_set) / len(relevant_set)

def demo():
    print("="*60)
    print("Демонстрация гибридного поиска и реранкинга с метриками")
    print("="*60)

    docs = [
        "Самозанятые граждане платят налог на профессиональный доход (НПД). Ставка 4% при работе с физлицами и 6% с юрлицами.",
        "Для российских IT-компаний с аккредитацией Минцифры налог на прибыль 0% до конца 2024, с 2025 — 5%.",
        "Страховые взносы для самозанятых необязательны, можно платить добровольно для пенсии.",
        "IT-компании освобождены от НДС при продаже собственного ПО.",
        "Налоговый кодекс РФ, статья 284.5 устанавливает пониженные ставки налога на прибыль для IT-компаний.",
        "ФЗ-422 о налоге на профессиональный доход регулирует ставки для самозанятых."
    ]
    metas = [
        {"source": "Закон о НПД (ФЗ-422)"},
        {"source": "НК РФ (Ст. 284.5)"},
        {"source": "Закон о НПД (Ст. 14)"},
        {"source": "НК РФ (Ст. 145.1)"},
        {"source": "НК РФ (Ст. 284.5)"},
        {"source": "ФЗ-422"}
    ]

    embed_model = SentenceTransformer("cointegrated/rubert-tiny2", device="cpu")
    llm = OllamaAdapter(MODEL_NAME)
    rag = RAGSystemAdvanced(llm, embed_model, docs, metas)

    # очистка и заполнение БД
    try:
        if rag.collection.count() > 0:
            ids = rag.collection.get()['ids']
            if ids:
                rag.collection.delete(ids)
    except:
        pass
    rag.add_documents(docs, metas)

    ground_truth = {
        "Какой налог платят самозанятые?": [0, 5],
        "Ставка налога на прибыль для IT-компаний?": [1, 4],
        "Что говорит статья 284.5 НК РФ?": [4],
        "Обязательны ли страховые взносы для самозанятых?": [2]
    }

    questions = list(ground_truth.keys())
    vec_recalls, hybrid_recalls, vec_times, hybrid_times = [], [], [], []

    for q in questions:
        print(f"\nВопрос: {q}")
        relevant = ground_truth[q]

        # векторный поиск
        start = time.time()
        q_emb = embed_model.encode([q], normalize_embeddings=True).tolist()
        vec_res = rag.collection.query(query_embeddings=q_emb, n_results=5)
        vec_time = time.time() - start
        vec_ids = [int(id.split('_')[1]) for id in vec_res['ids'][0]]
        recall_vec = compute_recall_at_k(vec_ids, relevant, k=5)
        vec_recalls.append(recall_vec)
        vec_times.append(vec_time)

        # гибридный + реранкинг
        start = time.time()
        hybrid_res = rag.search_only(q)
        hybrid_time = time.time() - start
        hybrid_ids = [doc['doc_idx'] for doc in hybrid_res]
        recall_hybrid = compute_recall_at_k(hybrid_ids, relevant, k=5)
        hybrid_recalls.append(recall_hybrid)
        hybrid_times.append(hybrid_time)

        print(f"  Векторный: Recall@5={recall_vec:.3f}, время={vec_time:.3f}с")
        print(f"  Гибрид+реранкинг: Recall@5={recall_hybrid:.3f}, время={hybrid_time:.3f}с")

    print("\n" + "="*60)
    print("Средние результаты:")
    print(f"  Векторный: Recall@5={np.mean(vec_recalls):.3f} ± {np.std(vec_recalls):.3f}, время={np.mean(vec_times):.3f}с")
    print(f"  Гибрид+реранкинг: Recall@5={np.mean(hybrid_recalls):.3f} ± {np.std(hybrid_recalls):.3f}, время={np.mean(hybrid_times):.3f}с")
    print("="*60)

if __name__ == "__main__":
    demo()

# Тема 3. Улучшение качества генерации

In [ ]:
# ================================================================
# Тема 3. Улучшение качества генерации (исправленная версия)
# Полный код для Google Colab
# ================================================================

!pip install -q requests sentence-transformers chromadb rank-bm25

import os
import sys
import time
import json
import re
import logging
import requests
import subprocess
import numpy as np
from typing import List, Dict, Tuple, Optional, Generator
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
print(f"Среда: {'Colab' if IS_COLAB else 'локально'}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DB_PATH = "/content/drive/MyDrive/chroma_db"
else:
    DB_PATH = "./chroma_db"

MODEL_NAME = "qwen2.5:1.5b"

# ========== Установка Ollama ==========
def ensure_model(model_name):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        if r.status_code == 200:
            models = [m['name'] for m in r.json().get('models', [])]
            if model_name in models:
                logger.info(f"Модель {model_name} уже загружена")
                return True
            else:
                logger.info(f"Загрузка {model_name}...")
                subprocess.run(["ollama", "pull", model_name], check=True, capture_output=True)
                logger.info(f"Модель {model_name} загружена")
                return True
    except Exception as e:
        logger.error(f"Ошибка: {e}")
        return False

def setup_ollama():
    if not IS_COLAB:
        try:
            subprocess.run(["ollama", "--version"], check=True, capture_output=True)
            return ensure_model(MODEL_NAME)
        except:
            logger.warning("Ollama не найден, установите вручную")
            return False
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        logger.info("Ollama уже запущен")
        return ensure_model(MODEL_NAME)
    except:
        pass

    logger.info("Установка Ollama...")
    !apt-get install -y zstd pciutils > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh
    os.environ["PATH"] += os.pathsep + "/usr/local/bin"
    try:
        subprocess.run(["ollama", "--version"], check=True, capture_output=True)
    except:
        logger.error("Ошибка установки")
        return False

    logger.info("Запуск сервера...")
    os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(30):
        try:
            requests.get("http://localhost:11434/api/tags", timeout=2)
            logger.info("Ollama готов")
            break
        except:
            time.sleep(1)
    else:
        logger.error("Сервер не запустился")
        return False
    return ensure_model(MODEL_NAME)

if not setup_ollama():
    sys.exit(1)

# ========== Адаптер Ollama ==========
def query_ollama(prompt, model=MODEL_NAME, temperature=0.7, max_tokens=512, system=None, retries=3, timeout=300):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": max_tokens}
    }
    if system:
        payload["system"] = system
    for attempt in range(retries):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json().get("response", "")
        except Exception as e:
            logger.warning(f"Попытка {attempt+1} ошибка: {e}")
            time.sleep(2 ** attempt)
    return "[Ошибка генерации]"

def query_ollama_stream(prompt, model=MODEL_NAME, temperature=0.7, system=None) -> Generator[str, None, None]:
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True,
        "options": {"temperature": temperature},
        "system": system
    }
    try:
        with requests.post(url, json=payload, stream=True, timeout=300) as resp:
            resp.raise_for_status()
            for line in resp.iter_lines():
                if line:
                    data = json.loads(line)
                    if 'response' in data:
                        yield data['response']
                    if data.get('done', False):
                        break
    except Exception as e:
        logger.error(f"Ошибка стриминга: {e}")
        yield f"[Ошибка: {e}]"

class OllamaAdapter:
    def __init__(self, model=MODEL_NAME):
        self.model = model
    def generate(self, prompt, system=None, temperature=0.7, max_tokens=512):
        return query_ollama(prompt, self.model, temperature, max_tokens, system)
    def generate_stream(self, prompt, system=None, temperature=0.7):
        return query_ollama_stream(prompt, self.model, temperature, system)

# ========== Ретривер ==========
class AdvancedRetriever:
    def __init__(self, collection, embed_model, texts, metadatas, alpha=0.6, k1=1.2, b=0.75,
                 rerank_top_k=20, final_top_k=5):
        self.collection = collection
        self.embed_model = embed_model
        self.texts = texts
        self.metadatas = metadatas
        self.alpha = alpha
        self.rerank_top_k = rerank_top_k
        self.final_top_k = final_top_k
        tokenized = [doc.split() for doc in texts]
        self.bm25 = BM25Okapi(tokenized, k1=k1, b=b)
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        logger.info("Cross-encoder загружен")

    def _vector_search(self, query, top_k):
        q_emb = self.embed_model.encode([query], normalize_embeddings=True).tolist()
        res = self.collection.query(query_embeddings=q_emb, n_results=top_k)
        ids = res['ids'][0]
        dist = res['distances'][0]
        sim = [1 - d for d in dist]
        return ids, sim

    def _bm25_search(self, query, top_k):
        tok = query.split()
        scores = self.bm25.get_scores(tok)
        indices = np.argsort(scores)[::-1][:top_k]
        return indices, scores

    def search(self, query, filter_metadata=None):
        vec_ids, vec_scores = self._vector_search(query, self.rerank_top_k * 2)
        bm25_indices, all_scores = self._bm25_search(query, self.rerank_top_k * 2)
        max_bm25 = max(all_scores) if all_scores.any() else 1.0

        combined = {}
        for idx, score in zip(vec_ids, vec_scores):
            doc_idx = int(idx.split('_')[1])
            combined[doc_idx] = self.alpha * score
        for doc_idx in bm25_indices:
            score = all_scores[doc_idx] / max_bm25 if max_bm25 > 0 else 0
            combined[doc_idx] = combined.get(doc_idx, 0) + (1 - self.alpha) * score

        sorted_items = sorted(combined.items(), key=lambda x: x[1], reverse=True)

        seen = set()
        unique = []
        for doc_idx, score in sorted_items:
            text = self.texts[doc_idx]
            if text not in seen:
                seen.add(text)
                unique.append((doc_idx, score))
        sorted_items = unique

        if filter_metadata:
            filtered = []
            for doc_idx, score in sorted_items:
                meta = self.metadatas[doc_idx]
                if all(meta.get(k) == v for k, v in filter_metadata.items()):
                    filtered.append((doc_idx, score))
            sorted_items = filtered

        candidates = sorted_items[:self.rerank_top_k]
        if not candidates:
            return []

        candidate_texts = [self.texts[idx] for idx, _ in candidates]
        pairs = [[query, doc] for doc in candidate_texts]
        rerank_scores = self.reranker.predict(pairs)

        final_indices = np.argsort(rerank_scores)[::-1][:self.final_top_k]
        results = []
        for i in final_indices:
            doc_idx = candidates[i][0]
            results.append({
                'doc_idx': doc_idx,
                'text': self.texts[doc_idx],
                'metadata': self.metadatas[doc_idx],
                'score': float(rerank_scores[i])
            })
        return results

# ========== RAG система (исправленная) ==========
class RAGSystemAdvanced:
    def __init__(self, llm_adapter, embed_model, texts, metadatas, db_path=DB_PATH):
        self.llm = llm_adapter
        self.embed_model = embed_model
        self.texts = texts
        self.metadatas = metadatas
        self.client = chromadb.PersistentClient(path=db_path)
        self.collection = None
        self._init_collection()
        self.retriever = AdvancedRetriever(
            self.collection, self.embed_model, texts, metadatas,
            alpha=0.6, rerank_top_k=20, final_top_k=5
        )
        self.default_system = (
            "Ты — эксперт-консультант. Отвечай на вопросы, используя ТОЛЬКО предоставленный контекст. "
            "Если в контексте нет информации, необходимой для ответа, скажи: «Я не знаю, в предоставленных документах нет ответа». "
            "Не добавляй информацию из своих знаний, если она не подтверждена контекстом. "
            "Всегда ссылайся на источники, указывая название документа. "
            "Отвечай структурированно, используй маркированные списки при перечислении."
        )

    def _init_collection(self):
        try:
            self.collection = self.client.get_collection("documents")
            logger.info(f"Загружено документов: {self.collection.count()}")
        except:
            self.collection = self.client.create_collection("documents")
            logger.info("Создана новая коллекция")

    def add_documents(self, documents, metadatas=None):
        if metadatas is None:
            metadatas = [{} for _ in documents]
        embeddings = self.embed_model.encode(documents, normalize_embeddings=True).tolist()
        ids = [f"doc_{i}" for i in range(len(documents))]
        self.collection.add(documents=documents, embeddings=embeddings, metadatas=metadatas, ids=ids)
        logger.info(f"Добавлено {len(documents)} документов")

    def _build_prompt(self, question, context, system_instruction=None):
        if system_instruction is None:
            system_instruction = self.default_system
        return f"<|system|>\n{system_instruction}\n\n<|context|>\n{context}\n\n<|question|>\n{question}\n\n<|answer|>"

    def _clean_response(self, text):
        text = re.sub(r'<\|.*?\|>', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _add_sources(self, response, sources):
        if not sources:
            return response
        sources_text = "\n\n**Источники:**\n" + "\n".join(f"- {s}" for s in sources)
        return response + sources_text

    def _check_faithfulness(self, answer, context, threshold=0.5):
        sentences = [s.strip() for s in answer.split('. ') if s]
        if not sentences:
            return True
        pairs = [[sent, context] for sent in sentences]
        scores = self.retriever.reranker.predict(pairs)
        faithful = sum(1 for s in scores if s > threshold) / len(scores)
        return faithful > 0.5

    def query(self, question, filter_metadata=None, system_prompt=None,
              return_sources=False, stream=False):
        # Поиск документов
        retrieved = self.retriever.search(question, filter_metadata=filter_metadata)
        if not retrieved:
            logger.warning(f"Документы не найдены для запроса: {question}")
            no_docs_msg = (
                "В предоставленных документах нет информации, отвечающей на ваш вопрос. "
                "Попробуйте переформулировать запрос, используя более конкретные термины."
            )
            if stream:
                def empty_gen():
                    yield no_docs_msg
                return empty_gen()
            return no_docs_msg

        # Формируем контекст и источники
        context = ""
        sources = []
        for i, doc in enumerate(retrieved):
            src = doc['metadata'].get('source', 'неизвестно')
            sources.append(src)
            context += f"\n[Документ {i+1}] Источник: {src}\n{doc['text']}\n"

        prompt = self._build_prompt(question, context, system_prompt)

        if stream:
            def gen():
                full = ""
                for token in self.llm.generate_stream(prompt, system=self.default_system, temperature=0.3):
                    full += token
                    yield token
                # Постобработка для потокового режима здесь не применима
            return gen()
        else:
            answer = self.llm.generate(prompt, system=self.default_system, temperature=0.3, max_tokens=512)
            answer = self._clean_response(answer)
            # Проверка согласованности (логируем, но не влияем на ответ)
            faithful = self._check_faithfulness(answer, context)
            if not faithful:
                logger.warning("Ответ может не соответствовать контексту (низкая согласованность)")
            if return_sources:
                answer = self._add_sources(answer, sources)
            return answer

# ========== Демонстрация ==========
def demo():
    print("="*60)
    print("Демонстрация улучшений генерации (промпт-инжиниринг, обработка no-docs, потоковая передача)")
    print("="*60)

    docs = [
        "Самозанятые граждане платят налог на профессиональный доход (НПД). Ставка 4% при работе с физлицами и 6% с юрлицами.",
        "Для российских IT-компаний с аккредитацией Минцифры налог на прибыль 0% до конца 2024, с 2025 — 5%.",
        "Страховые взносы для самозанятых необязательны, можно платить добровольно для пенсии.",
        "IT-компании освобождены от НДС при продаже собственного ПО.",
        "Налоговый кодекс РФ, статья 284.5 устанавливает пониженные ставки налога на прибыль для IT-компаний.",
        "ФЗ-422 о налоге на профессиональный доход регулирует ставки для самозанятых."
    ]
    metas = [
        {"source": "Закон о НПД (ФЗ-422)"},
        {"source": "НК РФ (Ст. 284.5)"},
        {"source": "Закон о НПД (Ст. 14)"},
        {"source": "НК РФ (Ст. 145.1)"},
        {"source": "НК РФ (Ст. 284.5)"},
        {"source": "ФЗ-422"}
    ]

    embed_model = SentenceTransformer("cointegrated/rubert-tiny2", device="cpu")
    llm = OllamaAdapter(MODEL_NAME)
    rag = RAGSystemAdvanced(llm, embed_model, docs, metas)

    # Очистка и заполнение БД
    try:
        if rag.collection.count() > 0:
            ids = rag.collection.get()['ids']
            if ids:
                rag.collection.delete(ids)
    except:
        pass
    rag.add_documents(docs, metas)

    questions = [
        "Какой налог платят самозанятые?",
        "Что говорит статья 284.5 НК РФ?",
        "Какой налог на имущество для физических лиц?"   # нет в контексте
    ]

    print("\n--- Обычный режим (без потоков) ---")
    for q in questions:
        print(f"\nВопрос: {q}")
        answer = rag.query(q, return_sources=True, stream=False)
        print(f"Ответ:\n{answer}")

    print("\n--- Потоковый режим ---")
    q = "Ставка налога на прибыль для IT-компаний?"
    print(f"Вопрос: {q}")
    print("Ответ (поток): ", end="")
    for token in rag.query(q, stream=True):
        print(token, end="", flush=True)
    print("\n")

    # Сравнение с системной инструкцией и без
    print("\n--- Сравнение с системной инструкцией и без ---")
    q = "Самозанятые платят налоги?"
    print(f"Вопрос: {q}")

    ans_no_sys = rag.llm.generate(
        f"Контекст:\n{docs[0]}\n\nВопрос: {q}\nОтвет:",
        system=None,
        temperature=0.3
    )
    print("Без системной инструкции:")
    print(ans_no_sys)

    ans_with_sys = rag.query(q, system_prompt=rag.default_system, return_sources=True, stream=False)
    print("\nС системной инструкцией (требовать источники):")
    print(ans_with_sys)

    print("\n" + "="*60)

if __name__ == "__main__":
    demo()

# Тема 4. Память и контекст в RAG (Полный код для Google Colab)

In [ ]:
# ================================================================
# Тема 4. Память и контекст в RAG (Полный код для Google Colab)
# ================================================================

!pip install -q requests sentence-transformers chromadb rank-bm25

import os
import sys
import time
import json
import re
import logging
import requests
import subprocess
import numpy as np
from typing import List, Dict, Tuple, Optional, Generator
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
print(f"Среда: {'Colab' if IS_COLAB else 'локально'}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DB_PATH = "/content/drive/MyDrive/chroma_db"
else:
    DB_PATH = "./chroma_db"

MODEL_NAME = "qwen2.5:1.5b"

# ========== Установка и запуск Ollama ==========
def ensure_model(model_name):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        if r.status_code == 200:
            models = [m['name'] for m in r.json().get('models', [])]
            if model_name in models:
                logger.info(f"Модель {model_name} уже загружена")
                return True
            else:
                logger.info(f"Загрузка {model_name}...")
                subprocess.run(["ollama", "pull", model_name], check=True, capture_output=True)
                logger.info(f"Модель {model_name} загружена")
                return True
    except Exception as e:
        logger.error(f"Ошибка: {e}")
        return False

def setup_ollama():
    if not IS_COLAB:
        try:
            subprocess.run(["ollama", "--version"], check=True, capture_output=True)
            return ensure_model(MODEL_NAME)
        except:
            logger.warning("Ollama не найден, установите вручную")
            return False
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        logger.info("Ollama уже запущен")
        return ensure_model(MODEL_NAME)
    except:
        pass

    logger.info("Установка Ollama...")
    !apt-get install -y zstd pciutils > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh
    os.environ["PATH"] += os.pathsep + "/usr/local/bin"
    try:
        subprocess.run(["ollama", "--version"], check=True, capture_output=True)
    except:
        logger.error("Ошибка установки")
        return False

    logger.info("Запуск сервера...")
    os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(30):
        try:
            requests.get("http://localhost:11434/api/tags", timeout=2)
            logger.info("Ollama готов")
            break
        except:
            time.sleep(1)
    else:
        logger.error("Сервер не запустился")
        return False
    return ensure_model(MODEL_NAME)

if not setup_ollama():
    sys.exit(1)

# ========== Адаптер Ollama ==========
def query_ollama(prompt, model=MODEL_NAME, temperature=0.7, max_tokens=512, system=None, retries=3, timeout=300):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": max_tokens}
    }
    if system:
        payload["system"] = system
    for attempt in range(retries):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json().get("response", "")
        except Exception as e:
            logger.warning(f"Попытка {attempt+1} ошибка: {e}")
            time.sleep(2 ** attempt)
    return "[Ошибка генерации]"

class OllamaAdapter:
    def __init__(self, model=MODEL_NAME):
        self.model = model
    def generate(self, prompt, system=None, temperature=0.7, max_tokens=512):
        return query_ollama(prompt, self.model, temperature, max_tokens, system)

# ========== Ретривер (гибридный + реранкинг) ==========
class AdvancedRetriever:
    def __init__(self, collection, embed_model, texts, metadatas, alpha=0.6, k1=1.2, b=0.75,
                 rerank_top_k=20, final_top_k=5):
        self.collection = collection
        self.embed_model = embed_model
        self.texts = texts
        self.metadatas = metadatas
        self.alpha = alpha
        self.rerank_top_k = rerank_top_k
        self.final_top_k = final_top_k
        tokenized = [doc.split() for doc in texts]
        self.bm25 = BM25Okapi(tokenized, k1=k1, b=b)
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        logger.info("Cross-encoder загружен")

    def _vector_search(self, query, top_k):
        q_emb = self.embed_model.encode([query], normalize_embeddings=True).tolist()
        res = self.collection.query(query_embeddings=q_emb, n_results=top_k)
        ids = res['ids'][0]
        dist = res['distances'][0]
        sim = [1 - d for d in dist]
        return ids, sim

    def _bm25_search(self, query, top_k):
        tok = query.split()
        scores = self.bm25.get_scores(tok)
        indices = np.argsort(scores)[::-1][:top_k]
        return indices, scores

    def search(self, query, filter_metadata=None):
        vec_ids, vec_scores = self._vector_search(query, self.rerank_top_k * 2)
        bm25_indices, all_scores = self._bm25_search(query, self.rerank_top_k * 2)
        max_bm25 = max(all_scores) if all_scores.any() else 1.0

        combined = {}
        for idx, score in zip(vec_ids, vec_scores):
            doc_idx = int(idx.split('_')[1])
            combined[doc_idx] = self.alpha * score
        for doc_idx in bm25_indices:
            score = all_scores[doc_idx] / max_bm25 if max_bm25 > 0 else 0
            combined[doc_idx] = combined.get(doc_idx, 0) + (1 - self.alpha) * score

        sorted_items = sorted(combined.items(), key=lambda x: x[1], reverse=True)

        seen = set()
        unique = []
        for doc_idx, score in sorted_items:
            text = self.texts[doc_idx]
            if text not in seen:
                seen.add(text)
                unique.append((doc_idx, score))
        sorted_items = unique

        if filter_metadata:
            filtered = []
            for doc_idx, score in sorted_items:
                meta = self.metadatas[doc_idx]
                if all(meta.get(k) == v for k, v in filter_metadata.items()):
                    filtered.append((doc_idx, score))
            sorted_items = filtered

        candidates = sorted_items[:self.rerank_top_k]
        if not candidates:
            return []

        candidate_texts = [self.texts[idx] for idx, _ in candidates]
        pairs = [[query, doc] for doc in candidate_texts]
        rerank_scores = self.reranker.predict(pairs)

        final_indices = np.argsort(rerank_scores)[::-1][:self.final_top_k]
        results = []
        for i in final_indices:
            doc_idx = candidates[i][0]
            results.append({
                'doc_idx': doc_idx,
                'text': self.texts[doc_idx],
                'metadata': self.metadatas[doc_idx],
                'score': float(rerank_scores[i])
            })
        return results

# ========== Управление памятью (история диалога) ==========
class ConversationMemory:
    def __init__(self, max_turns=5):
        self.history = []
        self.max_turns = max_turns

    def add_message(self, role, content):
        self.history.append({"role": role, "content": content})
        if len(self.history) > self.max_turns * 2:
            self.history = self.history[-self.max_turns * 2:]

    def get_history(self):
        return self.history

    def clear(self):
        self.history = []

    def get_history_text(self):
        return "\n".join([
            f"{'Пользователь' if msg['role'] == 'user' else 'Ассистент'}: {msg['content']}"
            for msg in self.history[-self.max_turns * 2:]
        ])

# ========== RAG система с памятью ==========
class RAGSystemWithMemory:
    def __init__(self, llm_adapter, embed_model, texts, metadatas, db_path=DB_PATH):
        self.llm = llm_adapter
        self.embed_model = embed_model
        self.texts = texts
        self.metadatas = metadatas

        self.client = chromadb.PersistentClient(path=db_path)
        self.collection = None
        self._init_collection()
        self.retriever = AdvancedRetriever(
            self.collection, self.embed_model, texts, metadatas,
            alpha=0.6, rerank_top_k=20, final_top_k=5
        )

        self.memory = ConversationMemory(max_turns=5)

        self.default_system = (
            "Ты — эксперт-консультант. Отвечай на вопросы, используя ТОЛЬКО предоставленный контекст. "
            "Если в контексте нет информации, скажи: «Я не знаю, в предоставленных документах нет ответа». "
            "Не добавляй информацию из своих знаний, если она не подтверждена контекстом. "
            "Всегда ссылайся на источники, указывая название документа. "
            "Учитывай историю диалога для понимания уточняющих вопросов."
        )

    def _init_collection(self):
        try:
            self.collection = self.client.get_collection("documents")
            logger.info(f"Загружено документов: {self.collection.count()}")
        except:
            self.collection = self.client.create_collection("documents")
            logger.info("Создана новая коллекция")

    def add_documents(self, documents, metadatas=None):
        if metadatas is None:
            metadatas = [{} for _ in documents]
        embeddings = self.embed_model.encode(documents, normalize_embeddings=True).tolist()
        ids = [f"doc_{i}" for i in range(len(documents))]
        self.collection.add(documents=documents, embeddings=embeddings, metadatas=metadatas, ids=ids)
        logger.info(f"Добавлено {len(documents)} документов")

    def _build_prompt(self, question, context, system_instruction=None):
        if system_instruction is None:
            system_instruction = self.default_system

        history_text = self.memory.get_history_text()
        history_section = f"<|history|>\n{history_text}\n\n" if history_text else ""

        return (f"<|system|>\n{system_instruction}\n\n"
                f"{history_section}"
                f"<|context|>\n{context}\n\n"
                f"<|question|>\n{question}\n\n"
                f"<|answer|>")

    def _clean_response(self, text):
        text = re.sub(r'<\|.*?\|>', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _add_sources(self, response, sources):
        if not sources:
            return response
        sources_text = "\n\n**Источники:**\n" + "\n".join(f"- {s}" for s in sources)
        return response + sources_text

    def _rewrite_query(self, question):
        """
        Переформулирует запрос с учётом истории.
        Использует эвристики + при необходимости LLM (закомментировано).
        """
        history = self.memory.get_history()
        if len(history) < 2:
            return question

        # Эвристика: если вопрос короткий или начинается с "А", "И", добавляем контекст
        if question.startswith(("А", "И", "а", "и")) or len(question.split()) < 4:
            last_user_msg = None
            for msg in reversed(history):
                if msg['role'] == 'user' and msg['content'] != question:
                    last_user_msg = msg['content']
                    break
            if last_user_msg:
                # Добавляем контекст из предыдущего вопроса
                return f"{last_user_msg} {question.lower()}"
        return question

    def query(self, question, system_prompt=None, return_sources=False):
        # Обработка команд
        if question.startswith("/"):
            if question in ("/reset", "/clear"):
                self.memory.clear()
                return "История диалога сброшена."

        # Добавляем вопрос в историю (для контекста при переформулировке)
        self.memory.add_message("user", question)

        # Переформулировка запроса
        rewritten = self._rewrite_query(question)
        if rewritten != question:
            logger.info(f"Переформулировка: '{question}' -> '{rewritten}'")

        # Поиск по переформулированному запросу
        retrieved = self.retriever.search(rewritten)
        if not retrieved:
            msg = "В предоставленных документах нет информации, отвечающей на ваш вопрос."
            self.memory.add_message("assistant", msg)
            return msg

        # Контекст и источники
        context = ""
        sources = []
        for i, doc in enumerate(retrieved):
            src = doc['metadata'].get('source', 'неизвестно')
            sources.append(src)
            context += f"\n[Документ {i+1}] Источник: {src}\n{doc['text']}\n"

        # Сборка промпта и генерация
        prompt = self._build_prompt(question, context, system_prompt)
        answer = self.llm.generate(prompt, system=self.default_system, temperature=0.3, max_tokens=512)
        answer = self._clean_response(answer)

        # Добавляем ответ в историю
        self.memory.add_message("assistant", answer)

        if return_sources:
            answer = self._add_sources(answer, sources)

        return answer

# ========== Демонстрация ==========
def demo():
    print("="*60)
    print("Демонстрация работы с памятью и контекстом")
    print("="*60)

    docs = [
        "Самозанятые граждане платят налог на профессиональный доход (НПД). Ставка 4% при работе с физлицами и 6% с юрлицами.",
        "Для российских IT-компаний с аккредитацией Минцифры налог на прибыль 0% до конца 2024, с 2025 — 5%.",
        "Страховые взносы для самозанятых необязательны, можно платить добровольно для пенсии.",
        "IT-компании освобождены от НДС при продаже собственного ПО.",
        "Налоговый кодекс РФ, статья 284.5 устанавливает пониженные ставки налога на прибыль для IT-компаний.",
        "ФЗ-422 о налоге на профессиональный доход регулирует ставки для самозанятых."
    ]
    metas = [
        {"source": "Закон о НПД (ФЗ-422)"},
        {"source": "НК РФ (Ст. 284.5)"},
        {"source": "Закон о НПД (Ст. 14)"},
        {"source": "НК РФ (Ст. 145.1)"},
        {"source": "НК РФ (Ст. 284.5)"},
        {"source": "ФЗ-422"}
    ]

    embed_model = SentenceTransformer("cointegrated/rubert-tiny2", device="cpu")
    llm = OllamaAdapter(MODEL_NAME)
    rag = RAGSystemWithMemory(llm, embed_model, docs, metas)

    # Очистка и заполнение БД
    try:
        if rag.collection.count() > 0:
            ids = rag.collection.get()['ids']
            if ids:
                rag.collection.delete(ids)
    except:
        pass
    rag.add_documents(docs, metas)

    dialog = [
        "Какие налоги платят самозанятые?",
        "А какие ставки?",
        "А льготы для них есть?",
        "Сравни с IT-компаниями",
        "/reset"
    ]

    print("\n--- Диалог с памятью ---")
    for q in dialog:
        print(f"\n👤 Пользователь: {q}")
        answer = rag.query(q, return_sources=True)
        print(f"🤖 Ассистент: {answer}")

    print("\n" + "="*60)

if __name__ == "__main__":
    demo()

`
